<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">
<br>
<h1 style="font-family:verdana; font-size:26px"><center>~ Datathon 2026 - Task 1: Peramalan Kecepatan Lalu Lintas ~</center></h1>
<h3 style="font-family:verdana; font-size:14px; color:gray"><center>Residual Graph-STID + Mean-Reversion Baseline | Horizon h5, h10, h15</center></h3>
<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Daftar Isi

1. [**Konfigurasi**](#1)
2. [**Inisialisasi**](#2)
3. [**Penemuan Path dan Validasi File**](#3)
4. [**Muat Data**](#4)
5. [**Profil Data dan Deteksi Regime**](#5)
6. [**Pembangunan Fold dan Pemeriksaan Kebocoran**](#6)
7. [**Baseline Mean-Reversion**](#7)
8. [**Normalisasi dan Prakomputasi Fitur Graf**](#8)
9. [**Dataset dan DataLoader**](#9)
10. [**Residual Graph-STID**](#10)
11. [**Compact Graph WaveNet**](#11)
12. [**Mesin Pelatihan**](#12)
13. [**Evaluasi MSE Skala Asli**](#13)
14. [**Eksekusi Smoke / CV**](#14)
15. [**Ringkasan Ranking dan Acceptance Gate**](#15)
16. [**Pelatihan Final**](#16)
17. [**Inferensi Test**](#17)
18. [**Ensemble OOF per Horizon**](#18)
19. [**Pembuatan Submission**](#19)
20. [**Ringkasan Hasil**](#20)

---

## Tujuan

Notebook ini melatih, memvalidasi, membandingkan, dan menghasilkan *submission* model prakiraan kecepatan lalu lintas tanpa mengubah *cell* selain *cell* konfigurasi.

**Arsitektur prediksi:**

```
prediksi_akhir = prediksi_mean_reversion + residual_model
```

Model utama adalah *Residual Graph-STID* yang menggabungkan riwayat kecepatan, agregasi graf, identitas ruas, regime, dan metadata jalan. *Compact Graph WaveNet* menjadi model penantang.

**Metrik resmi:** MSE pada skala kecepatan asli (km/jam).

**Batas validasi yang sah:**
- MSE rata-rata harus lebih kecil dari 39,00
- Model harus mengalahkan *mean-reversion* pada sekurang-kurangnya 2 dari 3 fold
- Seluruh prediksi harus *finite* dan berada dalam rentang `PREDICTION_CLIP`

In [1]:
# =============================================================
# CELL 1 - KONFIGURASI (satu-satunya cell yang boleh diedit)
# =============================================================
CFG = {
    # ---- Path ----
    "DATA_DIR": None,            # None = auto-discovery dari /kaggle/input atau ./dataset
    "WORK_DIR": "/kaggle/working",

    # ---- Mode eksekusi ----
    "RUN_MODE": "smoke",         # smoke | cv | final
    "MODEL": "graph_stid",       # graph_stid | graph_wavenet
    "FOLDS": [2],                # [2] untuk cepat, [0, 1, 2] untuk CV penuh

    # ---- Fitur ----
    "USE_GRAPH": True,
    "GRAPH_HOPS": 2,
    "USE_ROAD_METADATA": True,
    "USE_TEXT": False,

    # ---- Arsitektur ----
    "HIDDEN_DIM": 128,
    "ROAD_EMBED_DIM": 32,
    "TEXT_EMBED_DIM": 32,

    # ---- Training ----
    "BATCH_SIZE": 16,
    "EPOCHS": 40,
    "PATIENCE": 7,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,
    "DROPOUT": 0.10,
    "GRAD_CLIP": 1.0,
    "AMP": True,
    "SEED": 42,
    "NUM_WORKERS": 2,

    # ---- Inferensi ----
    "PREDICTION_CLIP": [0.0, 160.0],
    "MAKE_SUBMISSION": False,
}

In [2]:
# =============================================================
# CELL 3 - IMPORTS, SEED, GPU, VERSI
# =============================================================
import json
import math
import os
import platform
import random
import sys
import time
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import scipy.sparse as sp

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PERINGATAN: PyTorch tidak tersedia. Sel model akan dilewati.")

warnings.filterwarnings("ignore")


def set_seed(seed: int) -> None:
    """Mengunci seluruh sumber keacakan agar hasil dapat direproduksi."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(CFG["SEED"])

# Informasi environment
print("Python  :", sys.version.split()[0])
print("OS      :", platform.system(), platform.release())
print("numpy   :", np.__version__)
print("pandas  :", pd.__version__)

if TORCH_AVAILABLE:
    print("torch   :", torch.__version__)
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"GPU {i}   : {p.name} | VRAM {p.total_memory / 1e9:.1f} GB")
        DEVICE = torch.device("cuda")
    else:
        print("GPU     : tidak tersedia, memakai CPU")
        DEVICE = torch.device("cpu")
else:
    DEVICE = None

print("CWD     :", os.getcwd())

!nvidia-smi 2>/dev/null || echo "nvidia-smi tidak tersedia"

Python  : 3.12.13
OS      : Linux 6.12.90+
numpy   : 2.0.2
pandas  : 2.3.3
torch   : 2.10.0+cu128
GPU 0   : Tesla T4 | VRAM 15.6 GB
GPU 1   : Tesla T4 | VRAM 15.6 GB
CWD     : /kaggle/working
Sat Jul 18 06:14:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70

In [3]:
# =============================================================
# CELL 4 - PENEMUAN PATH OTOMATIS DAN VALIDASI FILE
# =============================================================

def discover_data_root(cfg_data_dir: Optional[str]) -> Path:
    """Menemukan direktori data secara otomatis berdasarkan environment.

    Urutan prioritas:
    1. CFG["DATA_DIR"] jika tidak None
    2. Path Kaggle standar
    3. Folder dataset/ lokal
    """
    if cfg_data_dir is not None:
        root = Path(cfg_data_dir)
        if root.exists():
            return root
        raise FileNotFoundError(f"DATA_DIR yang dikonfigurasi tidak ditemukan: {root}")

    # Coba path Kaggle
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        # Cari file test_X_hist.npy secara rekursif
        candidates = list(kaggle_root.rglob("test_X_hist.npy"))
        if candidates:
            # Gunakan direktori induk yang memuat folder train, test, static
            for c in candidates:
                parent = c.parent.parent
                if (parent / "train").exists() and (parent / "static").exists():
                    return parent
        raise FileNotFoundError(
            "Tidak menemukan struktur dataset di /kaggle/input. "
            "Pastikan dataset telah ditambahkan ke notebook."
        )

    # Fallback lokal
    local = Path("dataset")
    if local.exists():
        return local

    raise FileNotFoundError(
        "Tidak menemukan direktori data. Atur CFG['DATA_DIR'] secara eksplisit."
    )


DATA_ROOT = discover_data_root(CFG["DATA_DIR"])
WORK_DIR = Path(CFG["WORK_DIR"])
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "outputs").mkdir(exist_ok=True)

PATHS = {
    "train_speed_m1": DATA_ROOT / "train" / "train_speed_m1_1_11160.npy",
    "train_speed_m2": DATA_ROOT / "train" / "train_speed_m2_1_5039.npy",
    "train_text_m1": DATA_ROOT / "train" / "train_text_m1_1_11160.json",
    "train_text_m2": DATA_ROOT / "train" / "train_text_m2_1_5039.json",
    "test_hist": DATA_ROOT / "test" / "test_X_hist.npy",
    "test_texts": DATA_ROOT / "test" / "test_texts.json",
    "adjacency": DATA_ROOT / "static" / "matrix.npy",
    "roads_meta": DATA_ROOT / "static" / "Roads1260.json",
    "active_mask": DATA_ROOT / "static" / "active_mask.npy",
    "sample_sub": DATA_ROOT / "sample_submission.csv",
}

print(f"Environment : {'Kaggle' if Path('/kaggle/input').exists() else 'Lokal'}")
print(f"DATA_ROOT   : {DATA_ROOT}")
print(f"WORK_DIR    : {WORK_DIR}")
print()
missing = []
for name, p in PATHS.items():
    status = "OK    " if p.exists() else "HILANG"
    print(f"  {name:18s} {status}  {p}")
    if not p.exists():
        missing.append(name)

if missing:
    raise FileNotFoundError(f"File tidak ditemukan: {missing}")
print("\nSeluruh file terverifikasi.")

Environment : Kaggle
DATA_ROOT   : /kaggle/input/competitions/datathon-task-1/dataset-task1
WORK_DIR    : /kaggle/working

  train_speed_m1     OK      /kaggle/input/competitions/datathon-task-1/dataset-task1/train/train_speed_m1_1_11160.npy
  train_speed_m2     OK      /kaggle/input/competitions/datathon-task-1/dataset-task1/train/train_speed_m2_1_5039.npy
  train_text_m1      OK      /kaggle/input/competitions/datathon-task-1/dataset-task1/train/train_text_m1_1_11160.json
  train_text_m2      OK      /kaggle/input/competitions/datathon-task-1/dataset-task1/train/train_text_m2_1_5039.json
  test_hist          OK      /kaggle/input/competitions/datathon-task-1/dataset-task1/test/test_X_hist.npy
  test_texts         OK      /kaggle/input/competitions/datathon-task-1/dataset-task1/test/test_texts.json
  adjacency          OK      /kaggle/input/competitions/datathon-task-1/dataset-task1/static/matrix.npy
  roads_meta         OK      /kaggle/input/competitions/datathon-task-1/dataset-task1

In [4]:
# =============================================================
# CELL 5 - MUAT DATA
# =============================================================

def _ordered_json(d: dict) -> list:
    """Mengurutkan nilai dict berdasarkan indeks numerik pada kunci."""
    return [d[k] for k in sorted(d, key=lambda s: int(s.split("_")[-1]))]


t0 = time.time()

# Kecepatan
speed_m1 = np.load(PATHS["train_speed_m1"])   # (11160, 1260)
speed_m2 = np.load(PATHS["train_speed_m2"])   # (5039, 1260)
test_hist = np.load(PATHS["test_hist"])        # (540, 15, 1260)

# Graf dan metadata
adj_raw = np.load(PATHS["adjacency"]).astype(np.float32)  # (1260, 1260)
np.fill_diagonal(adj_raw, 0)
active_mask = np.load(PATHS["active_mask"])
roads_meta = json.load(open(PATHS["roads_meta"]))
sample_sub = pd.read_csv(PATHS["sample_sub"])

# Teks
raw_text_m1 = json.load(open(PATHS["train_text_m1"]))
raw_text_m2 = json.load(open(PATHS["train_text_m2"]))
raw_text_te = json.load(open(PATHS["test_texts"]))
texts_m1 = _ordered_json(raw_text_m1)
texts_m2 = _ordered_json(raw_text_m2)
texts_te = _ordered_json(raw_text_te)
del raw_text_m1, raw_text_m2, raw_text_te

N_ROADS = 1260
HIST_LEN = 15
HORIZONS = (5, 10, 15)
N_TEST = 540

# Verifikasi shape
assert speed_m1.shape == (11160, N_ROADS), f"Shape M1 tidak sesuai: {speed_m1.shape}"
assert speed_m2.shape == (5039, N_ROADS), f"Shape M2 tidak sesuai: {speed_m2.shape}"
assert test_hist.shape == (N_TEST, HIST_LEN, N_ROADS), f"Shape test tidak sesuai: {test_hist.shape}"
assert adj_raw.shape == (N_ROADS, N_ROADS), f"Shape matrix tidak sesuai: {adj_raw.shape}"
assert np.isfinite(speed_m1).all(), "Nilai non-finite pada speed_m1"
assert np.isfinite(speed_m2).all(), "Nilai non-finite pada speed_m2"
assert np.isfinite(test_hist).all(), "Nilai non-finite pada test_hist"
assert len(texts_m1) == len(speed_m1), "Panjang teks M1 tidak selaras"
assert len(texts_m2) == len(speed_m2), "Panjang teks M2 tidak selaras"
assert len(texts_te) == N_TEST, "Panjang teks test tidak selaras"
assert len(roads_meta) == N_ROADS, f"Jumlah segmen tidak sesuai: {len(roads_meta)}"

print(f"Muat data selesai dalam {time.time() - t0:.1f} dtk")
for name, a in [("speed_m1", speed_m1), ("speed_m2", speed_m2), ("test_hist", test_hist)]:
    print(f"  {name:12s} shape={str(a.shape):20s} dtype={str(a.dtype):8s} "
          f"min={a.min():.2f} max={a.max():.2f}")
print(f"  adjacency     shape={str(adj_raw.shape):20s} tepi={int(adj_raw.sum())}")
print(f"  sample_sub    shape={str(sample_sub.shape):20s}")
print(f"  teks: m1={len(texts_m1)}, m2={len(texts_m2)}, test={len(texts_te)}")

Muat data selesai dalam 3.4 dtk
  speed_m1     shape=(11160, 1260)        dtype=float32  min=0.00 max=142.00
  speed_m2     shape=(5039, 1260)         dtype=float32  min=0.00 max=151.00
  test_hist    shape=(540, 15, 1260)      dtype=float32  min=0.00 max=137.00
  adjacency     shape=(1260, 1260)         tepi=3862
  sample_sub    shape=(2041200, 2)        
  teks: m1=11160, m2=5039, test=540


In [5]:
# =============================================================
# CELL 6 - PROFIL DATA DAN DETEKSI REGIME
# =============================================================

ZERO_FRAC_M1 = float((speed_m1 == 0).mean())
ZERO_FRAC_M2 = float((speed_m2 == 0).mean())
ZERO_FRAC_TEST = float((test_hist == 0).mean())

print("Profil data:")
print(f"  M1 : {speed_m1.shape[0]} timestep, fraksi nol = {ZERO_FRAC_M1:.4f} ({ZERO_FRAC_M1:.1%})")
print(f"  M2 : {speed_m2.shape[0]} timestep, fraksi nol = {ZERO_FRAC_M2:.4f} ({ZERO_FRAC_M2:.1%})")
print(f"  Test: {N_TEST} sampel, fraksi nol = {ZERO_FRAC_TEST:.4f} ({ZERO_FRAC_TEST:.1%})")

# Prototipe regime berdasarkan fraksi nol per sampel (pada window 15 timestep)
# Setiap sampel test memiliki window 15 x 1260 - hitung fraksi nol per sampel
test_zero_frac_per_sample = (test_hist == 0).mean(axis=(1, 2))  # (540,)

# Threshold berdasarkan prototipe dari train
# M1 ~ 1% nol, M2 ~ 16.7% nol -> threshold di tengah = ~8-9%
PROTO_M1 = ZERO_FRAC_M1    # ~0.01
PROTO_M2 = ZERO_FRAC_M2    # ~0.167
THRESHOLD_REGIME = (PROTO_M1 + PROTO_M2) / 2.0


def infer_regimes(test_windows: np.ndarray, prototypes: dict) -> np.ndarray:
    """Menetapkan regime (m1 atau m2) untuk setiap sampel test berdasarkan fraksi nol.

    Args:
        test_windows: array shape (N, HIST, ROADS)
        prototypes: dict {"m1": float, "m2": float} fraksi nol prototipe

    Returns:
        array string dengan nilai "m1" atau "m2" untuk setiap sampel.
    """
    zero_fracs = (test_windows == 0).mean(axis=(1, 2))
    threshold = (prototypes["m1"] + prototypes["m2"]) / 2.0
    return np.where(zero_fracs < threshold, "m1", "m2")


prototypes = {"m1": PROTO_M1, "m2": PROTO_M2}
test_regimes = infer_regimes(test_hist, prototypes)

N_M1_TEST = int((test_regimes == "m1").sum())
N_M2_TEST = int((test_regimes == "m2").sum())

print(f"\nKlasifikasi regime test:")
print(f"  M1: {N_M1_TEST} sampel")
print(f"  M2: {N_M2_TEST} sampel")
print(f"  Total: {N_M1_TEST + N_M2_TEST} sampel")

assert N_M1_TEST == 372, f"Jumlah sampel M1 test diharapkan 372, dapat {N_M1_TEST}"
assert N_M2_TEST == 168, f"Jumlah sampel M2 test diharapkan 168, dapat {N_M2_TEST}"
print("\nAssersi regime test lulus: 372 M1 / 168 M2.")

# Mask ruas yang selalu nol per regime
always_zero_m1 = (speed_m1 == 0).all(axis=0)  # (1260,) bool
always_zero_m2 = (speed_m2 == 0).all(axis=0)  # (1260,) bool
print(f"\nRuas yang selalu nol: M1={always_zero_m1.sum()}, M2={always_zero_m2.sum()}")

# Bobot regime untuk evaluasi (berdasarkan distribusi test)
REGIME_WEIGHTS = {"m1": N_M1_TEST / N_TEST, "m2": N_M2_TEST / N_TEST}  # 372/540, 168/540
print(f"Bobot evaluasi: {REGIME_WEIGHTS}")

Profil data:
  M1 : 11160 timestep, fraksi nol = 0.0104 (1.0%)
  M2 : 5039 timestep, fraksi nol = 0.1667 (16.7%)
  Test: 540 sampel, fraksi nol = 0.0594 (5.9%)

Klasifikasi regime test:
  M1: 372 sampel
  M2: 168 sampel
  Total: 540 sampel

Assersi regime test lulus: 372 M1 / 168 M2.

Ruas yang selalu nol: M1=13, M2=210
Bobot evaluasi: {'m1': 0.6888888888888889, 'm2': 0.3111111111111111}


In [6]:
# =============================================================
# CELL 7 - PEMBANGUNAN FOLD DAN PEMERIKSAAN KEBOCORAN
# =============================================================

# Definisi fold berbasis waktu (berlaku per regime):
# Fold 0: train [0%, 55%],   val [55%, 70%]
# Fold 1: train [0%, 70%],   val [70%, 85%]
# Fold 2: train [0%, 85%],   val [85%, 100%]
FOLD_BOUNDS = [
    (0.0, 0.55, 0.55, 0.70),
    (0.0, 0.70, 0.70, 0.85),
    (0.0, 0.85, 0.85, 1.00),
]


def make_windows(
    speed: np.ndarray,
    origins: np.ndarray,
    history: int = 15,
    horizons: tuple = (5, 10, 15),
) -> Tuple[np.ndarray, np.ndarray]:
    """Membentuk jendela riwayat dan target dari deret kecepatan.

    Args:
        speed: array (T, N_ROADS)
        origins: indeks timestep akhir riwayat, shape (W,)
        history: panjang riwayat
        horizons: tuple horizon target

    Returns:
        histories: (W, history, N_ROADS)
        targets: (W, len(horizons), N_ROADS)
    """
    W = len(origins)
    idx_hist = origins[:, None] - np.arange(history - 1, -1, -1)[None, :]  # (W, history)
    histories = speed[idx_hist]  # (W, history, N_ROADS)

    target_list = []
    for h in horizons:
        target_list.append(speed[origins + h])  # (W, N_ROADS)
    targets = np.stack(target_list, axis=1)  # (W, len(horizons), N_ROADS)

    return histories, targets


def build_fold_origins(
    n_timesteps: int,
    train_end_frac: float,
    val_start_frac: float,
    val_end_frac: float,
    history: int = 15,
    max_horizon: int = 15,
) -> Tuple[np.ndarray, np.ndarray]:
    """Menghasilkan indeks origin untuk train dan validation pada satu fold.

    Indeks origin adalah timestep terakhir dari jendela riwayat.
    Seluruh riwayat validasi berada setelah batas train.
    """
    min_origin = history - 1
    max_origin = n_timesteps - max_horizon - 1

    train_end_idx = int(n_timesteps * train_end_frac) - 1
    val_start_idx = int(n_timesteps * val_start_frac) + history - 1
    val_end_idx = int(n_timesteps * val_end_frac) - max_horizon

    train_origins = np.arange(min_origin, min(train_end_idx, max_origin) + 1)
    val_origins = np.arange(
        max(val_start_idx, min_origin),
        min(val_end_idx, max_origin) + 1,
    )

    # Pemeriksaan kebocoran: riwayat validasi harus sepenuhnya setelah batas train
    if len(val_origins) > 0:
        earliest_val_history = val_origins[0] - (history - 1)
        assert earliest_val_history >= int(n_timesteps * train_end_frac), (
            f"KEBOCORAN TERDETEKSI: riwayat validasi ({earliest_val_history}) "
            f"tumpang tindih dengan train (batas={int(n_timesteps * train_end_frac)})"
        )

    return train_origins, val_origins


# Uji fold untuk setiap regime
FOLDS_M1, FOLDS_M2 = [], []
for i, (tr0, tr1, va0, va1) in enumerate(FOLD_BOUNDS):
    tr_m1, va_m1 = build_fold_origins(len(speed_m1), tr1, va0, va1)
    tr_m2, va_m2 = build_fold_origins(len(speed_m2), tr1, va0, va1)
    FOLDS_M1.append((tr_m1, va_m1))
    FOLDS_M2.append((tr_m2, va_m2))
    print(f"Fold {i}: "
          f"M1 train={len(tr_m1)} val={len(va_m1)} | "
          f"M2 train={len(tr_m2)} val={len(va_m2)}")

print("\nSeluruh pemeriksaan kebocoran fold lulus.")

Fold 0: M1 train=6124 val=1645 | M2 train=2757 val=728
Fold 1: M1 train=7797 val=1647 | M2 train=3513 val=728
Fold 2: M1 train=9472 val=1645 | M2 train=4269 val=727

Seluruh pemeriksaan kebocoran fold lulus.


In [7]:
# =============================================================
# CELL 8 - BASELINE MEAN-REVERSION
# =============================================================

def compute_mean_reversion_coeffs(
    speed: np.ndarray,
    train_origins: np.ndarray,
    history: int = 15,
    horizons: tuple = (5, 10, 15),
    stride: int = 30,
) -> Dict[int, np.ndarray]:
    """Menghitung koefisien mean-reversion per ruas per horizon dari data train.

    Koefisien c dipilih agar prediksi = nilai_terakhir + c * (mean_long - nilai_terakhir)
    meminimalkan MSE pada sub-sampel train.

    Args:
        speed: array (T, N_ROADS)
        train_origins: indeks origin train
        history: panjang riwayat
        stride: langkah sub-sampel untuk efisiensi komputasi

    Returns:
        dict {horizon: array koefisien (N_ROADS,)}
    """
    # Hitung mean jangka panjang per ruas dari seluruh data train
    train_speed = speed[: train_origins[-1] + 1]
    valid = np.where(train_speed > 0, train_speed, np.nan)
    long_mean = np.nanmean(valid, axis=0)  # (N_ROADS,)
    long_mean = np.nan_to_num(long_mean, nan=0.0)

    coeffs = {}
    for h in horizons:
        origins_sub = train_origins[::stride]
        last_vals = speed[origins_sub]                   # (W, N_ROADS)
        targets = speed[origins_sub + h]                 # (W, N_ROADS)
        delta = long_mean[None, :] - last_vals           # (W, N_ROADS)

        # OLS per ruas: c = sum(delta * (target - last_val)) / sum(delta^2)
        num = (delta * (targets - last_vals)).sum(axis=0)
        den = (delta ** 2).sum(axis=0)
        c = np.where(den > 1e-8, num / den, 0.0)
        c = np.clip(c, 0.0, 1.0)
        coeffs[h] = c.astype(np.float32)

    return long_mean.astype(np.float32), coeffs


def predict_mean_reversion(
    histories: np.ndarray,
    long_mean: np.ndarray,
    coeffs: Dict[int, np.ndarray],
    horizons: tuple = (5, 10, 15),
) -> np.ndarray:
    """Menghasilkan prediksi mean-reversion.

    Args:
        histories: (W, history, N_ROADS)
        long_mean: (N_ROADS,)
        coeffs: {horizon: (N_ROADS,)}

    Returns:
        predictions: (W, len(horizons), N_ROADS)
    """
    last_val = histories[:, -1, :]  # (W, N_ROADS)
    preds = []
    for h in horizons:
        c = coeffs[h][None, :]  # (1, N_ROADS)
        p = last_val + c * (long_mean[None, :] - last_val)
        preds.append(p)
    return np.stack(preds, axis=1)  # (W, len(horizons), N_ROADS)


def compose_prediction(
    baseline: np.ndarray,
    residual: np.ndarray,
) -> np.ndarray:
    """Menggabungkan prediksi baseline dan residual model.

    Args:
        baseline: (W, N_horizons, N_ROADS) atau (B, N_ROADS, N_horizons)
        residual: sama dengan baseline

    Returns:
        prediksi akhir dengan shape yang sama
    """
    return baseline + residual


# Hitung dan tampilkan skor baseline per fold shadow
print("=" * 60)
print("BASELINE MEAN-REVERSION")
print("=" * 60)

baseline_scores = []
for fold_idx in range(3):
    tr_m1, va_m1 = FOLDS_M1[fold_idx]
    tr_m2, va_m2 = FOLDS_M2[fold_idx]

    # Hitung koefisien dari data train saja
    mean_m1, c_m1 = compute_mean_reversion_coeffs(speed_m1, tr_m1, stride=30)
    mean_m2, c_m2 = compute_mean_reversion_coeffs(speed_m2, tr_m2, stride=30)

    mse_per_horizon = []
    for h_idx, h in enumerate(HORIZONS):
        # Evaluasi pada validasi M1
        hist_m1_va, tgt_m1_va = make_windows(speed_m1, va_m1)
        pred_m1 = predict_mean_reversion(hist_m1_va, mean_m1, c_m1)
        err_m1 = ((pred_m1[:, h_idx, :] - tgt_m1_va[:, h_idx, :]) ** 2).mean()

        # Evaluasi pada validasi M2
        hist_m2_va, tgt_m2_va = make_windows(speed_m2, va_m2)
        pred_m2 = predict_mean_reversion(hist_m2_va, mean_m2, c_m2)
        err_m2 = ((pred_m2[:, h_idx, :] - tgt_m2_va[:, h_idx, :]) ** 2).mean()

        # MSE gabungan dengan bobot regime
        mse_combined = REGIME_WEIGHTS["m1"] * err_m1 + REGIME_WEIGHTS["m2"] * err_m2
        mse_per_horizon.append(float(mse_combined))

    mean_mse = float(np.mean(mse_per_horizon))
    baseline_scores.append(mean_mse)
    print(f"Fold {fold_idx}: MSE per horizon {[f'{v:.4f}' for v in mse_per_horizon]} | "
          f"rata-rata = {mean_mse:.4f}")

global_baseline_mse = float(np.mean(baseline_scores))
print(f"\nMSE mean-reversion (rata-rata 3 fold): {global_baseline_mse:.4f}")

# Parity check: harus mendekati 40-42
# Jika jauh di luar rentang, pipeline tidak selaras
if global_baseline_mse < 35 or global_baseline_mse > 55:
    print(f"\nPERINGATAN: MSE baseline {global_baseline_mse:.4f} berada di luar rentang "
          f"yang diharapkan (35-55). Periksa target alignment.")
else:
    print("Baseline parity check: LULUS (MSE dalam rentang wajar).")

BASELINE MEAN-REVERSION
Fold 0: MSE per horizon ['38.8053', '46.7365', '51.9756'] | rata-rata = 45.8391
Fold 1: MSE per horizon ['35.1984', '41.7125', '46.0751'] | rata-rata = 40.9953
Fold 2: MSE per horizon ['36.3948', '41.2705', '44.3457'] | rata-rata = 40.6703

MSE mean-reversion (rata-rata 3 fold): 42.5016
Baseline parity check: LULUS (MSE dalam rentang wajar).


In [8]:
# =============================================================
# CELL 9 - NORMALISASI DAN PRAKOMPUTASI FITUR GRAF
# =============================================================

def row_normalize(adj: np.ndarray) -> np.ndarray:
    """Normalisasi baris matriks adjacency.

    Baris yang seluruhnya nol tetap nol (tidak ada pembagi nol).

    Args:
        adj: (N, N) float32

    Returns:
        matriks ternormalisasi baris (N, N) float32
    """
    row_sums = adj.sum(axis=1)
    row_sums = np.where(row_sums == 0, 1.0, row_sums)  # hindari pembagi nol
    return (adj / row_sums[:, None]).astype(np.float32)


def compute_graph_features(
    speed: np.ndarray,
    origins: np.ndarray,
    A_fwd_hops: List[np.ndarray],
    A_bwd_hops: List[np.ndarray],
    history: int = 15,
) -> np.ndarray:
    """Prakomputasi fitur agregasi graf untuk setiap jendela.

    Menghasilkan fitur agregasi tetangga maju dan mundur untuk hop 1 dan 2.
    Pemrosesan dilakukan dengan matriks sparse untuk efisiensi memori.

    Args:
        speed: (T, N_ROADS)
        origins: (W,) indeks timestep akhir riwayat
        A_fwd_hops: list matriks forward per hop, masing-masing (N, N)
        A_bwd_hops: list matriks backward per hop, masing-masing (N, N)
        history: panjang riwayat

    Returns:
        graph_feats: (W, history, N_ROADS * n_hops * 2) float32
    """
    W = len(origins)
    n_hops = len(A_fwd_hops)
    hist_idx = origins[:, None] - np.arange(history - 1, -1, -1)[None, :]  # (W, H)
    windows = speed[hist_idx]  # (W, H, N)

    feat_parts = []
    for hop_idx in range(n_hops):
        A_f = sp.csr_matrix(A_fwd_hops[hop_idx])
        A_b = sp.csr_matrix(A_bwd_hops[hop_idx])
        # Agregasi: (W, H, N) -> untuk setiap timestep, matmul (N,) @ A^T
        fwd = np.stack([windows[:, t, :] @ A_f.T.toarray() for t in range(history)], axis=1)
        bwd = np.stack([windows[:, t, :] @ A_b.T.toarray() for t in range(history)], axis=1)
        feat_parts.extend([fwd, bwd])

    return np.concatenate(feat_parts, axis=2).astype(np.float32)  # (W, H, N * n_hops * 2)


class SpeedNormalizer:
    """Normalisasi z-score per ruas, hanya dari data train."""

    def __init__(self, eps: float = 1.0):
        self.eps = eps
        self.mean_: Optional[np.ndarray] = None
        self.std_: Optional[np.ndarray] = None

    def fit(self, speed: np.ndarray) -> "SpeedNormalizer":
        """Fit dari array (T, N_ROADS), mengabaikan nilai nol."""
        valid = np.where(speed > 0, speed, np.nan)
        self.mean_ = np.nan_to_num(np.nanmean(valid, axis=0), nan=0.0).astype(np.float32)
        self.std_ = np.nan_to_num(np.nanstd(valid, axis=0), nan=1.0).astype(np.float32)
        self.std_ = np.maximum(self.std_, self.eps)
        return self

    def transform(self, speed: np.ndarray) -> np.ndarray:
        """Normalisasi, nilai nol tetap nol."""
        result = (speed - self.mean_) / self.std_
        result = np.where(speed == 0, 0.0, result)
        return result.astype(np.float32)

    def inverse_transform(self, normalized: np.ndarray, regime_mean: np.ndarray = None) -> np.ndarray:
        """Kembalikan ke skala asli."""
        mean = regime_mean if regime_mean is not None else self.mean_
        return (normalized * self.std_ + mean).astype(np.float32)


# Bangun matriks adjacency ternormalisasi forward dan backward
A_fwd_1 = row_normalize(adj_raw)
A_bwd_1 = row_normalize(adj_raw.T)

# Hop 2: perkalian hop 1 dengan dirinya sendiri
A_fwd_2 = row_normalize(A_fwd_1 @ A_fwd_1)
A_bwd_2 = row_normalize(A_bwd_1 @ A_bwd_1)

A_FWD_HOPS = [A_fwd_1, A_fwd_2] if CFG["USE_GRAPH"] and CFG["GRAPH_HOPS"] >= 2 else [A_fwd_1]
A_BWD_HOPS = [A_bwd_1, A_bwd_2] if CFG["USE_GRAPH"] and CFG["GRAPH_HOPS"] >= 2 else [A_bwd_1]
N_HOPS = len(A_FWD_HOPS)

print(f"Graf: {N_HOPS} hop maju dan mundur")
print(f"A_fwd_1 sum row rata-rata: {A_fwd_1.sum(1).mean():.3f}")
print(f"A_bwd_1 sum row rata-rata: {A_bwd_1.sum(1).mean():.3f}")

# Metadata jalan (1260 ruas)
if CFG["USE_ROAD_METADATA"]:
    road_len = np.array([sum(s["length"] for s in r) for r in roads_meta], dtype=np.float32)
    road_nseg = np.array([len(r) for r in roads_meta], dtype=np.float32)
    # Distribusi roadclass (per-ruas nilai unik pertama)
    road_class = np.array([r[0].get("roadclass", 0) for r in roads_meta], dtype=np.float32)
    road_formway = np.array([r[0].get("formway", 0) for r in roads_meta], dtype=np.float32)

    # Normalisasi metadata statis
    def _norm_stat(x: np.ndarray) -> np.ndarray:
        mu, sigma = x.mean(), x.std() + 1e-6
        return ((x - mu) / sigma).astype(np.float32)

    road_static = np.stack([
        _norm_stat(road_len),
        _norm_stat(road_nseg),
        _norm_stat(road_class),
        _norm_stat(road_formway),
    ], axis=1)  # (1260, 4)
    N_STATIC = road_static.shape[1]
    print(f"Metadata statis: {road_static.shape}")
else:
    road_static = np.zeros((N_ROADS, 0), dtype=np.float32)
    N_STATIC = 0

print("Normalisasi dan prakomputasi graf selesai.")

Graf: 2 hop maju dan mundur
A_fwd_1 sum row rata-rata: 0.990
A_bwd_1 sum row rata-rata: 0.990
Metadata statis: (1260, 4)
Normalisasi dan prakomputasi graf selesai.


In [9]:
# =============================================================
# CELL 10 - DATASET LAZY DAN DATALOADER
# =============================================================

KEYWORDS = ['road closure', 'accident', 'construction', 'traffic control', 'prohibit', 'announcement']

def extract_text_features(texts: list, keywords: list) -> np.ndarray:
    T, K, N = len(texts), len(keywords), 1260
    feats = np.zeros((T, K), dtype=np.float32)
    for i, t in enumerate(texts):
        for j, kw in enumerate(keywords):
            feats[i, j] = t.count(kw)
    return np.broadcast_to(feats[:, None, :], (T, N, K))

class LazyTrafficDataset(Dataset):
    """Dataset berbasis jendela untuk peramalan kecepatan lalu lintas.

    Membangun fitur secara on-the-fly untuk menghemat memori.
    Setiap item berisi tensor fitur dinamis, baseline mean-reversion,
    target (pada skala asli), dan mask ruas aktif.

    Args:
        speed_raw: (T, N_ROADS) kecepatan asli
        origins: (W,) indeks timestep akhir riwayat
        normalizer: SpeedNormalizer yang sudah di-fit
        long_mean: (N_ROADS,) rata-rata jangka panjang untuk mean-reversion
        mr_coeffs: {h: (N_ROADS,)} koefisien mean-reversion
        A_fwd: list matriks forward per hop
        A_bwd: list matriks backward per hop
        road_static_feats: (N_ROADS, S) fitur statis
        horizons: tuple horizon
        history: panjang riwayat
        max_samples: batasi jumlah sampel (untuk mode smoke)
    """

    def __init__(
        self,
        speed_raw: np.ndarray,
        origins: np.ndarray,
        normalizer: SpeedNormalizer,
        long_mean: np.ndarray,
        mr_coeffs: Dict[int, np.ndarray],
        A_fwd: List[np.ndarray],
        A_bwd: List[np.ndarray],
        road_static_feats: np.ndarray,
        text_feats = None,
        horizons: tuple = (5, 10, 15),
        history: int = 15,
        max_samples: Optional[int] = None,
    ):
        self.speed_raw = speed_raw
        self.normalizer = normalizer
        self.long_mean = long_mean
        self.mr_coeffs = mr_coeffs
        self.A_fwd = A_fwd
        self.A_bwd = A_bwd
        self.road_static = road_static_feats
        self.text_feats = text_feats
        self.horizons = horizons
        self.history = history

        self.origins = origins
        if max_samples is not None and len(origins) > max_samples:
            # Ambil distribusi merata agar representasi temporal terjaga
            idx = np.linspace(0, len(origins) - 1, max_samples, dtype=int)
            self.origins = origins[idx]

        # Prakomputasi matriks sparse dalam format yang efisien
        self._A_fwd_sp = [sp.csr_matrix(A) for A in A_fwd]
        self._A_bwd_sp = [sp.csr_matrix(A) for A in A_bwd]

    def __len__(self) -> int:
        return len(self.origins)

    def __getitem__(self, idx: int) -> Dict[str, "torch.Tensor"]:
        origin = self.origins[idx]
        h_len = self.history

        # Ambil jendela riwayat (history, N_ROADS)
        hist_idx = np.arange(origin - h_len + 1, origin + 1)
        window_raw = self.speed_raw[hist_idx]  # (H, N)

        # Normalisasi riwayat
        window_norm = self.normalizer.transform(window_raw)  # (H, N)

        # Agregasi tetangga (fitur dinamis tambahan)
        n_hops = len(self._A_fwd_sp)
        hop_feats = []
        for A_f, A_b in zip(self._A_fwd_sp, self._A_bwd_sp):
            fwd = window_norm @ A_f.T.toarray()  # (H, N)
            bwd = window_norm @ A_b.T.toarray()  # (H, N)
            hop_feats.extend([fwd, bwd])

        # Statistik temporal per ruas (scalar features)
        last_val = window_norm[-1]          # (N,)
        mean3 = window_norm[-3:].mean(0)    # (N,)
        mean5 = window_norm[-5:].mean(0)    # (N,)
        mean15 = window_norm.mean(0)        # (N,)
        diff1 = window_norm[-1] - window_norm[-2]  # (N,)
        diff5 = window_norm[-1] - window_norm[-5]  # (N,) jika history>=5
        std5 = window_norm[-5:].std(0)     # (N,)
        std15 = window_norm.std(0)         # (N,)
        # Konteks jaringan global
        global_mean = mean15.mean() * np.ones(self.speed_raw.shape[1], dtype=np.float32)

        # Gabungkan fitur dinamis:
        # [last_val, mean3, mean5, mean15, diff1, diff5, std5, std15, global_mean]
        # + hop aggregations
        scalar_feats = np.stack(
            [last_val, mean3, mean5, mean15, diff1, diff5, std5, std15, global_mean],
            axis=0  # (9, N)
        )  # (9, N)

        # Semua fitur dinamis: (H_total, N)
        # Kita gabungkan riwayat + hop feats dalam dimensi fitur per-ruas
        # Output akhir: (N, D_dynamic) untuk setiap ruas
        hop_arr = np.stack(hop_feats, axis=0)  # (n_hops*2, H, N)
        # Ratakan hop per timestep terakhir saja
        hop_last = hop_arr[:, -1, :]  # (n_hops*2, N)

        dynamic_feats = np.concatenate(
            [scalar_feats, hop_last], axis=0
        ).T  # (N, 9 + n_hops*2)

        # Baseline mean-reversion
        last_raw = window_raw[-1]  # (N,)
        baseline_preds = []
        for h in self.horizons:
            c = self.mr_coeffs[h]
            p = last_raw + c * (self.long_mean - last_raw)
            baseline_preds.append(p)
        baseline = np.stack(baseline_preds, axis=1)  # (N, len(horizons))

        # Target pada skala asli
        targets = np.stack(
            [self.speed_raw[origin + h] for h in self.horizons], axis=1
        ).astype(np.float32)  # (N, len(horizons))

        # Mask ruas aktif (tidak selalu nol dalam window ini)
        active_mask = (window_raw > 0).any(axis=0)  # (N,)

        return {
            "dynamic": torch.from_numpy(dynamic_feats.astype(np.float32)),  # (N, D)
            "baseline": torch.from_numpy(baseline.astype(np.float32)),       # (N, H)
            "target": torch.from_numpy(targets.astype(np.float32)),           # (N, H)
            "active_mask": torch.from_numpy(active_mask),                     # (N,) bool
        }


def make_dataloaders(
    speed: np.ndarray,
    train_origins: np.ndarray,
    val_origins: np.ndarray,
    normalizer: SpeedNormalizer,
    long_mean: np.ndarray,
    mr_coeffs: dict,
    cfg: dict,
    max_train_samples: Optional[int] = None,
    text_feats = None,
) -> Tuple[DataLoader, DataLoader]:
    """Membangun DataLoader train dan validasi untuk satu fold."""
    ds_train = LazyTrafficDataset(
        speed_raw=speed,
        origins=train_origins,
        normalizer=normalizer,
        long_mean=long_mean,
        mr_coeffs=mr_coeffs,
        A_fwd=A_FWD_HOPS,
        A_bwd=A_BWD_HOPS,
        road_static_feats=road_static,
        text_feats=text_feats,
        horizons=HORIZONS,
        history=HIST_LEN,
        max_samples=max_train_samples,
    )
    ds_val = LazyTrafficDataset(
        speed_raw=speed,
        origins=val_origins,
        normalizer=normalizer,
        long_mean=long_mean,
        mr_coeffs=mr_coeffs,
        A_fwd=A_FWD_HOPS,
        A_bwd=A_BWD_HOPS,
        road_static_feats=road_static,
        text_feats=text_feats,
        horizons=HORIZONS,
        history=HIST_LEN,
        max_samples=None,
    )
    loader_train = DataLoader(
        ds_train,
        batch_size=cfg["BATCH_SIZE"],
        shuffle=True,
        num_workers=cfg["NUM_WORKERS"],
        pin_memory=DEVICE.type == "cuda" if DEVICE else False,
        drop_last=True,
    )
    loader_val = DataLoader(
        ds_val,
        batch_size=cfg["BATCH_SIZE"] * 2,
        shuffle=False,
        num_workers=cfg["NUM_WORKERS"],
        pin_memory=DEVICE.type == "cuda" if DEVICE else False,
    )
    return loader_train, loader_val


# Tentukan dimensi input
_n_hops = len(A_FWD_HOPS)
DYNAMIC_DIM = 9 + _n_hops * 2  # scalar feats + hop aggregations
STATIC_DIM = N_STATIC
print(f"DYNAMIC_DIM = {DYNAMIC_DIM} | STATIC_DIM = {STATIC_DIM}")

DYNAMIC_DIM = 13 | STATIC_DIM = 4


In [10]:
# =============================================================
# CELL 11 - RESIDUAL GRAPH-STID
# =============================================================

class ResidualMLP(nn.Module):
    """Blok MLP residual dengan LayerNorm, GELU, dan dropout."""

    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim * 2)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(dim * 2, dim)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x: "torch.Tensor") -> "torch.Tensor":
        # x: (..., dim)
        h = self.norm(x)
        h = self.fc2(self.drop(self.act(self.fc1(h))))
        return x + self.drop2(h)


class ResidualGraphSTID(nn.Module):
    """Residual Graph-STID untuk peramalan kecepatan lalu lintas.

    Prediksi akhir = baseline_mean_reversion + output_head(features)
    Head output diinisialisasi nol sehingga pada awal pelatihan
    keluaran model identik dengan baseline.

    Args:
        roads: jumlah ruas (1260)
        dynamic_dim: dimensi fitur dinamis per ruas
        static_dim: dimensi fitur statis per ruas
        hidden_dim: dimensi tersembunyi
        road_embed_dim: dimensi embedding ruas
        horizons: jumlah horizon prediksi
        dropout: laju dropout
    """

    def __init__(
        self,
        roads: int,
        dynamic_dim: int,
        static_dim: int,
        hidden_dim: int,
        road_embed_dim: int,
        horizons: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.roads = roads
        self.horizons = horizons

        # Embedding identitas ruas (learnable)
        self.road_embed = nn.Embedding(roads, road_embed_dim)

        # Proyeksi input ke hidden_dim
        input_dim = dynamic_dim + road_embed_dim + max(static_dim, 1)
        self.input_proj = nn.Linear(input_dim, hidden_dim)

        # Tiga blok MLP residual
        self.mlp_blocks = nn.ModuleList([
            ResidualMLP(hidden_dim, dropout) for _ in range(3)
        ])

        # Head output per horizon, diinisialisasi nol
        self.output_head = nn.Linear(hidden_dim, horizons)
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

        # Indeks ruas (buffer, tidak ikut gradient)
        self.register_buffer("road_ids", torch.arange(roads))

    def forward(
        self,
        dynamic: "torch.Tensor",
        static: "torch.Tensor",
        baseline: "torch.Tensor",
    ) -> "torch.Tensor":
        """
        Args:
            dynamic: (B, N, D_dyn) fitur dinamis
            static: (N, S) fitur statis (tidak ada dimensi batch)
            baseline: (B, N, H) prediksi baseline mean-reversion

        Returns:
            prediction: (B, N, H) prediksi akhir
        """
        B, N, _ = dynamic.shape

        # Embedding ruas
        road_emb = self.road_embed(self.road_ids)  # (N, E)
        road_emb = road_emb.unsqueeze(0).expand(B, -1, -1)  # (B, N, E)

        # Gabungkan fitur
        if static.shape[-1] > 0:
            static_exp = static.unsqueeze(0).expand(B, -1, -1)  # (B, N, S)
            x = torch.cat([dynamic, road_emb, static_exp], dim=-1)  # (B, N, D+E+S)
        else:
            # Jika static_dim=0, tambahkan dummy
            dummy = torch.zeros(B, N, 1, device=dynamic.device)
            x = torch.cat([dynamic, road_emb, dummy], dim=-1)  # (B, N, D+E+1)

        # Proyeksi ke hidden_dim
        x = self.input_proj(x)  # (B, N, H_dim)

        # MLP residual blocks
        for block in self.mlp_blocks:
            x = block(x)

        # Output residual
        residual = self.output_head(x)  # (B, N, horizons)

        # Prediksi akhir = baseline + residual
        return baseline + residual


print("ResidualGraphSTID didefinisikan.")
if TORCH_AVAILABLE:
    # Quick shape test
    _model_test = ResidualGraphSTID(
        roads=8, dynamic_dim=13, static_dim=4,
        hidden_dim=32, road_embed_dim=8, horizons=3, dropout=0.0
    )
    _dyn = torch.randn(2, 8, 13)
    _sta = torch.randn(8, 4)
    _base = torch.randn(2, 8, 3)
    _out = _model_test(_dyn, _sta, _base)
    assert _out.shape == (2, 8, 3), f"Shape salah: {_out.shape}"
    # Cek inisialisasi nol: keluaran awal harus sama dengan baseline
    torch.testing.assert_close(_out, _base, msg="Inisialisasi nol gagal: prediksi != baseline")
    print("Shape test ResidualGraphSTID: LULUS")
    print("Inisialisasi nol (prediksi == baseline): LULUS")
    del _model_test, _dyn, _sta, _base, _out

ResidualGraphSTID didefinisikan.
Shape test ResidualGraphSTID: LULUS
Inisialisasi nol (prediksi == baseline): LULUS


In [11]:
# =============================================================
# CELL 12 - COMPACT GRAPH WAVENET
# =============================================================

class DiffusionConv(nn.Module):
    """Konvolusi difusi graf orde-2 dengan adjacency maju dan mundur.

    Menggabungkan informasi tetangga forward (arah lalu lintas masuk)
    dan backward (arah lalu lintas keluar) secara terpisah.
    """

    def __init__(self, in_channels: int, out_channels: int, order: int = 2):
        super().__init__()
        self.order = order
        # Parameter untuk setiap urutan difusi: 2 arah x (order+1) hop
        n_matrices = 2 * (order + 1)
        self.weights = nn.Parameter(
            torch.FloatTensor(n_matrices, in_channels, out_channels)
        )
        nn.init.xavier_uniform_(self.weights)
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(
        self,
        x: "torch.Tensor",
        A_fwd: "torch.Tensor",
        A_bwd: "torch.Tensor",
    ) -> "torch.Tensor":
        """
        Args:
            x: (B, C_in, N) fitur node
            A_fwd: (N, N) adjacency maju ternormalisasi
            A_bwd: (N, N) adjacency mundur ternormalisasi

        Returns:
            out: (B, C_out, N)
        """
        B, C, N = x.shape
        out = torch.zeros(B, self.weights.shape[-1], N, device=x.device)
        mat_idx = 0
        for A in [A_fwd, A_bwd]:
            x_hop = x  # (B, C, N)
            for k in range(self.order + 1):
                out = out + torch.einsum("bcn,nk,cko->bko", x_hop, A, self.weights[mat_idx])
                mat_idx += 1
                if k < self.order:
                    x_hop = torch.einsum("bcn,nk->bck", x_hop, A)
        return out + self.bias


class GatedTemporalConv(nn.Module):
    """Konvolusi temporal bergerbang dengan dilasi."""

    def __init__(self, channels: int, kernel_size: int = 2, dilation: int = 1):
        super().__init__()
        self.conv = nn.Conv1d(
            channels, channels * 2,
            kernel_size=kernel_size, dilation=dilation,
            padding=0,
        )
        self.dilation = dilation
        self.kernel_size = kernel_size

    def forward(self, x: "torch.Tensor") -> "torch.Tensor":
        """
        Args:
            x: (B, C, T)

        Returns:
            out: (B, C, T - (kernel_size-1)*dilation)
        """
        out = self.conv(x)
        gate = torch.sigmoid(out[:, :out.size(1)//2])
        tanh = torch.tanh(out[:, out.size(1)//2:])
        return gate * tanh


class CompactGraphWaveNet(nn.Module):
    """Compact Graph WaveNet untuk peramalan kecepatan lalu lintas.

    Menggabungkan dilated temporal convolution dan diffusion graph convolution.
    Prediksi akhir = baseline + residual (sama dengan Graph-STID).

    Args:
        roads: jumlah ruas
        history: panjang riwayat
        horizons: jumlah horizon
        dilations: tuple dilasi temporal
        residual_channels: kanal residual
        skip_channels: kanal skip
    """

    def __init__(
        self,
        roads: int,
        history: int = 15,
        horizons: int = 3,
        dilations: Tuple[int, ...] = (1, 2, 4, 8),
        residual_channels: int = 32,
        skip_channels: int = 64,
    ):
        super().__init__()
        self.roads = roads
        self.horizons = horizons
        self.dilations = dilations

        # Receptive field
        self._receptive_field = 1 + sum((d * (2 - 1)) for d in dilations)

        # Proyeksi input (kecepatan 1 channel -> residual_channels)
        self.input_proj = nn.Conv2d(1, residual_channels, kernel_size=(1, 1))

        # Layer WaveNet
        self.temporal_convs = nn.ModuleList()
        self.graph_convs = nn.ModuleList()
        self.skip_convs = nn.ModuleList()
        self.residual_convs = nn.ModuleList()

        for d in dilations:
            self.temporal_convs.append(
                GatedTemporalConv(residual_channels, kernel_size=2, dilation=d)
            )
            self.graph_convs.append(
                DiffusionConv(residual_channels, residual_channels, order=2)
            )
            self.skip_convs.append(
                nn.Conv1d(residual_channels, skip_channels, kernel_size=1)
            )
            self.residual_convs.append(
                nn.Conv1d(residual_channels, residual_channels, kernel_size=1)
            )

        # Head output
        self.output_head = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(skip_channels, skip_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(skip_channels, horizons, kernel_size=1),
        )
        # Inisialisasi nol
        for m in self.output_head.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.zeros_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    @property
    def receptive_field(self) -> int:
        return self._receptive_field

    def forward(
        self,
        x: "torch.Tensor",
        baseline: "torch.Tensor",
        A_fwd: "torch.Tensor" = None,
        A_bwd: "torch.Tensor" = None,
    ) -> "torch.Tensor":
        """
        Args:
            x: (B, 1, N, T) kecepatan dinormalisasi
            baseline: (B, N, H) prediksi baseline
            A_fwd: (N, N) adjacency maju (opsional)
            A_bwd: (N, N) adjacency mundur (opsional)

        Returns:
            prediction: (B, N, H)
        """
        B, _, N, T = x.shape
        if A_fwd is None:
            A_fwd = torch.eye(N, device=x.device)
        if A_bwd is None:
            A_bwd = torch.eye(N, device=x.device)

        # Input proj: (B, C, N, T)
        h = self.input_proj(x)

        skip_sum = None
        for temp_conv, graph_conv, skip_conv, res_conv in zip(
            self.temporal_convs, self.graph_convs,
            self.skip_convs, self.residual_convs
        ):
            h_in = h
            # Temporal conv pada dimensi T: (B, C, N, T) -> ambil tiap node
            # Reshape ke (B*N, C, T) untuk Conv1d
            BN, C, Nt = h.view(B * N, -1, h.shape[-1]).shape
            h_t = h.view(B * N, C, -1)
            h_t = temp_conv(h_t)  # (B*N, C, T')
            h = h_t.view(B, N, C, -1).permute(0, 2, 1, 3)  # (B, C, N, T')

            # Graph conv pada timestep terakhir: (B, C, N)
            h_last = h[:, :, :, -1]  # (B, C, N)
            h_g = graph_conv(h_last, A_fwd, A_bwd)  # (B, C, N)

            # Skip
            s = skip_conv(h_g)  # (B, skip_channels, N)
            skip_sum = s if skip_sum is None else skip_sum[..., -s.shape[-1]:] + s

            # Residual - sesuaikan panjang temporal jika berbeda
            h_res = res_conv(h_g).unsqueeze(-1)  # (B, C, N, 1)
            if h_in.shape[-1] > h_res.shape[-1]:
                # Pangkas h_in untuk kecocokan dimensi
                h_res = h_res.expand(-1, -1, -1, h_in.shape[-1])
            h = h + h_res[..., :h.shape[-1]]

        # Output
        out = self.output_head(skip_sum)  # (B, horizons, N)
        residual = out.permute(0, 2, 1)   # (B, N, horizons)

        return baseline + residual


print("CompactGraphWaveNet didefinisikan.")
if TORCH_AVAILABLE:
    _wn_test = CompactGraphWaveNet(
        roads=8, history=15, horizons=3,
        dilations=(1, 2, 4, 8), residual_channels=8, skip_channels=16
    )
    assert _wn_test.receptive_field >= 15, (
        f"Receptive field ({_wn_test.receptive_field}) kurang dari history (15)"
    )
    print(f"Receptive field: {_wn_test.receptive_field} (>= 15): LULUS")
    del _wn_test

CompactGraphWaveNet didefinisikan.
Receptive field: 16 (>= 15): LULUS


In [12]:
# =============================================================
# CELL 13 - MESIN PELATIHAN
# =============================================================

@dataclass
class TrainingResult:
    """Hasil pelatihan satu model pada satu fold."""
    history: List[Dict[str, float]] = field(default_factory=list)
    best_val_mse: float = float("inf")
    best_epoch: int = 0
    checkpoint_path: str = ""


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    static_features: "torch.Tensor",
    epochs: int,
    patience: int,
    amp: bool,
    learning_rate: float,
    weight_decay: float,
    grad_clip: float,
    checkpoint_path,
    resume: bool = False,
    clip_range: Tuple[float, float] = (0.0, 160.0),
) -> TrainingResult:
    """Melatih model dengan AdamW, AMP, cosine LR decay, dan early stopping.

    Args:
        model: model PyTorch
        train_loader: DataLoader train
        val_loader: DataLoader validasi
        static_features: (N, S) tensor fitur statis ruas
        epochs: jumlah epoch maksimum
        patience: jumlah epoch tanpa peningkatan sebelum berhenti
        amp: aktifkan Automatic Mixed Precision
        learning_rate: learning rate AdamW
        weight_decay: weight decay AdamW
        grad_clip: norma maksimum gradient clipping
        checkpoint_path: path file checkpoint model terbaik
        resume: muat checkpoint jika tersedia
        clip_range: rentang kliping prediksi untuk evaluasi

    Returns:
        TrainingResult dengan riwayat loss dan informasi checkpoint
    """
    checkpoint_path = Path(checkpoint_path)
    model = model.to(DEVICE)
    static_features = static_features.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=learning_rate * 0.01
    )
    scaler = torch.cuda.amp.GradScaler() if (amp and DEVICE.type == "cuda") else None

    result = TrainingResult(checkpoint_path=str(checkpoint_path))
    start_epoch = 0
    no_improve = 0

    # Resume jika diminta
    if resume and checkpoint_path.exists():
        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        if scaler and "scaler_state" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"] + 1
        result.best_val_mse = ckpt["best_val_mse"]
        result.best_epoch = ckpt["best_epoch"]
        result.history = ckpt.get("history", [])
        no_improve = ckpt.get("no_improve", 0)
        print(f"Melanjutkan dari epoch {start_epoch}, best MSE={result.best_val_mse:.4f}")

    for epoch in range(start_epoch, epochs):
        # --- Train ---
        model.train()
        train_loss_sum = 0.0
        n_batches = 0
        t_start = time.time()

        for batch in train_loader:
            dynamic = batch["dynamic"].to(DEVICE)    # (B, N, D)
            baseline = batch["baseline"].to(DEVICE)  # (B, N, H)
            target = batch["target"].to(DEVICE)      # (B, N, H)
            active = batch["active_mask"].to(DEVICE) # (B, N)

            optimizer.zero_grad()

            ctx = torch.amp.autocast(device_type=DEVICE.type) if scaler else torch.no_grad.__class__()
            if scaler:
                with torch.amp.autocast(device_type=DEVICE.type):
                    pred = model(dynamic, static_features, baseline)
                    # MSE hanya pada ruas aktif, skala asli (target sudah dalam km/jam)
                    mask = active.unsqueeze(-1).float()  # (B, N, 1)
                    loss = ((pred - target) ** 2 * mask).sum() / (mask.sum() * pred.shape[-1] + 1e-8)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                pred = model(dynamic, static_features, baseline)
                mask = active.unsqueeze(-1).float()
                loss = ((pred - target) ** 2 * mask).sum() / (mask.sum() * pred.shape[-1] + 1e-8)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            train_loss_sum += loss.item()
            n_batches += 1

        scheduler.step()
        train_loss = train_loss_sum / max(n_batches, 1)

        # --- Validasi ---
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                dynamic = batch["dynamic"].to(DEVICE)
                baseline = batch["baseline"].to(DEVICE)
                target = batch["target"].to(DEVICE)
                pred = model(dynamic, static_features, baseline)
                val_preds.append(pred.cpu().numpy())
                val_targets.append(target.cpu().numpy())

        val_pred_np = np.concatenate(val_preds, axis=0)   # (W, N, H)
        val_tgt_np = np.concatenate(val_targets, axis=0)  # (W, N, H)

        # Kliping prediksi ke rentang fisik
        val_pred_np = np.clip(val_pred_np, clip_range[0], clip_range[1])

        val_mse = float(np.mean((val_pred_np - val_tgt_np) ** 2))
        epoch_time = time.time() - t_start

        step = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_mse": val_mse,
            "lr": scheduler.get_last_lr()[0],
            "time_s": epoch_time,
        }
        result.history.append(step)

        improved = val_mse < result.best_val_mse
        marker = " *" if improved else ""
        print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | "
              f"val_mse={val_mse:.4f}{marker} | "
              f"lr={scheduler.get_last_lr()[0]:.6f} | {epoch_time:.1f}s")

        if improved:
            result.best_val_mse = val_mse
            result.best_epoch = epoch
            no_improve = 0
            # Simpan checkpoint
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict() if scaler else {},
                "best_val_mse": result.best_val_mse,
                "best_epoch": result.best_epoch,
                "history": result.history,
                "no_improve": no_improve,
                "cfg": CFG,
                "seed": CFG["SEED"],
            }, checkpoint_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping pada epoch {epoch} (no_improve={no_improve})")
                break

    # Muat bobot terbaik
    if checkpoint_path.exists():
        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        print(f"Model terbaik dimuat dari epoch {result.best_epoch}, "
              f"val_mse={result.best_val_mse:.4f}")

    return result


print("Mesin pelatihan didefinisikan.")

Mesin pelatihan didefinisikan.


In [13]:
# =============================================================
# CELL 14 - EVALUASI MSE SKALA ASLI
# =============================================================

def evaluate_mse(
    model: nn.Module,
    speed_raw: np.ndarray,
    val_origins: np.ndarray,
    normalizer: SpeedNormalizer,
    long_mean: np.ndarray,
    mr_coeffs: dict,
    cfg: dict,
    regime_label: str,
    always_zero_mask: np.ndarray,
) -> Dict[str, float]:
    """Mengevaluasi MSE pada skala kecepatan asli (km/jam).

    Menghitung MSE keseluruhan, per horizon, dan melaporkan
    dibandingkan dengan baseline mean-reversion pada split yang sama.

    Returns:
        dict dengan kunci: overall, h5, h10, h15, baseline_overall
    """
    model.eval()
    model_preds = []    # (W, N, H)
    mr_preds = []       # (W, N, H)
    all_targets = []    # (W, N, H)

    ds = LazyTrafficDataset(
        speed_raw=speed_raw,
        origins=val_origins,
        normalizer=normalizer,
        long_mean=long_mean,
        mr_coeffs=mr_coeffs,
        A_fwd=A_FWD_HOPS,
        A_bwd=A_BWD_HOPS,
        road_static_feats=road_static,
        horizons=HORIZONS,
        history=HIST_LEN,
    )
    loader = DataLoader(
        ds,
        batch_size=cfg["BATCH_SIZE"] * 2,
        shuffle=False,
        num_workers=0,  # untuk evaluasi gunakan 0 worker
    )

    static_t = torch.from_numpy(road_static).to(DEVICE)

    with torch.no_grad():
        for batch in loader:
            dyn = batch["dynamic"].to(DEVICE)
            bl = batch["baseline"].to(DEVICE)
            tgt = batch["target"]

            pred = model(dyn, static_t, bl)  # (B, N, H)
            pred_np = pred.cpu().numpy()
            bl_np = batch["baseline"].numpy()

            # Kliping ke rentang fisik
            pred_np = np.clip(pred_np, cfg["PREDICTION_CLIP"][0], cfg["PREDICTION_CLIP"][1])

            # Terapkan mask ruas yang selalu nol
            pred_np[:, always_zero_mask, :] = 0.0
            bl_np[:, always_zero_mask, :] = 0.0

            model_preds.append(pred_np)
            mr_preds.append(bl_np)
            all_targets.append(tgt.numpy())

    model_preds = np.concatenate(model_preds, axis=0)    # (W, N, H)
    mr_preds = np.concatenate(mr_preds, axis=0)          # (W, N, H)
    all_targets = np.concatenate(all_targets, axis=0)    # (W, N, H)

    results = {}
    results["overall"] = float(np.mean((model_preds - all_targets) ** 2))
    results["baseline_overall"] = float(np.mean((mr_preds - all_targets) ** 2))

    for h_idx, h in enumerate(HORIZONS):
        results[f"h{h}"] = float(
            np.mean((model_preds[:, :, h_idx] - all_targets[:, :, h_idx]) ** 2)
        )
        results[f"h{h}_baseline"] = float(
            np.mean((mr_preds[:, :, h_idx] - all_targets[:, :, h_idx]) ** 2)
        )

    print(f"  [{regime_label}] Model MSE: {results['overall']:.4f} | "
          f"Baseline: {results['baseline_overall']:.4f} | "
          f"Delta: {results['overall'] - results['baseline_overall']:+.4f}")
    for h in HORIZONS:
        print(f"    h{h}: {results[f'h{h}']:.4f} (baseline: {results[f'h{h}_baseline']:.4f})")

    return results


print("Fungsi evaluasi MSE skala asli didefinisikan.")

Fungsi evaluasi MSE skala asli didefinisikan.


In [14]:
# =============================================================
# CELL 15 - EKSEKUSI SMOKE / CV
# =============================================================

import json as _json

RUN_MODE = CFG["RUN_MODE"]
MODEL_NAME = CFG["MODEL"]

# Batas jumlah sampel untuk mode smoke
SMOKE_MAX_SAMPLES = 512
SMOKE_EPOCHS = 2

# Penentu fold yang akan dijalankan
FOLDS_TO_RUN = CFG["FOLDS"] if RUN_MODE != "smoke" else [max(CFG["FOLDS"])]

print(f"Mode: {RUN_MODE} | Model: {MODEL_NAME} | Folds: {FOLDS_TO_RUN}")
print("=" * 60)

# Penyimpan hasil
cv_results = []            # list hasil per fold
oof_predictions = {}       # {fold_idx: (W, N, H) array prediksi OOF}
best_epochs_per_fold = []  # untuk menentukan epoch final

for fold_idx in FOLDS_TO_RUN:
    print(f"\n{'=' * 60}")
    print(f"FOLD {fold_idx}")
    print(f"{'=' * 60}")

    # ---- Setup per fold ----
    tr_m1, va_m1 = FOLDS_M1[fold_idx]
    tr_m2, va_m2 = FOLDS_M2[fold_idx]

    max_train = SMOKE_MAX_SAMPLES if RUN_MODE == "smoke" else None
    epochs = SMOKE_EPOCHS if RUN_MODE == "smoke" else CFG["EPOCHS"]

    # Fit normalizer dan koefisien mean-reversion HANYA dari train
    norm_m1 = SpeedNormalizer().fit(speed_m1[: tr_m1[-1] + 1])
    norm_m2 = SpeedNormalizer().fit(speed_m2[: tr_m2[-1] + 1])

    mean_m1, c_m1 = compute_mean_reversion_coeffs(speed_m1, tr_m1, stride=30)
    mean_m2, c_m2 = compute_mean_reversion_coeffs(speed_m2, tr_m2, stride=30)

    # Dataloader M1
    loader_train_m1, loader_val_m1 = make_dataloaders(
        speed_m1, tr_m1, va_m1, norm_m1, mean_m1, c_m1, CFG,
        max_train_samples=max_train // 2 if max_train else None,
    )
    # Dataloader M2
    loader_train_m2, loader_val_m2 = make_dataloaders(
        speed_m2, tr_m2, va_m2, norm_m2, mean_m2, c_m2, CFG,
        max_train_samples=max_train // 2 if max_train else None,
    )

    # Gabungkan loader ke iterator bergantian
    from itertools import chain

    def interleave_loaders(l1, l2):
        """Iterator yang bergantian mengambil dari dua loader."""
        for b1, b2 in zip(l1, l2):
            yield b1
            yield b2

    # Dataset gabungan untuk model tunggal (bobot bersama)
    from torch.utils.data import ConcatDataset

    # Buat dataset gabungan
    ds_train_m1 = LazyTrafficDataset(
        speed_m1, tr_m1, norm_m1, mean_m1, c_m1,
        A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN,
        max_samples=max_train // 2 if max_train else None,
    )
    ds_train_m2 = LazyTrafficDataset(
        speed_m2, tr_m2, norm_m2, mean_m2, c_m2,
        A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN,
        max_samples=max_train // 2 if max_train else None,
    )
    ds_val_m1 = LazyTrafficDataset(
        speed_m1, va_m1, norm_m1, mean_m1, c_m1,
        A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN,
    )
    ds_val_m2 = LazyTrafficDataset(
        speed_m2, va_m2, norm_m2, mean_m2, c_m2,
        A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN,
    )

    ds_train_all = ConcatDataset([ds_train_m1, ds_train_m2])
    ds_val_all = ConcatDataset([ds_val_m1, ds_val_m2])

    loader_train = DataLoader(
        ds_train_all, batch_size=CFG["BATCH_SIZE"], shuffle=True,
        num_workers=CFG["NUM_WORKERS"], pin_memory=DEVICE.type == "cuda", drop_last=True,
    )
    loader_val = DataLoader(
        ds_val_all, batch_size=CFG["BATCH_SIZE"] * 2, shuffle=False,
        num_workers=CFG["NUM_WORKERS"],
    )

    print(f"Train windows: {len(ds_train_all)} | Val windows: {len(ds_val_all)}")

    # ---- Bangun model ----
    static_t = torch.from_numpy(road_static).to(DEVICE)

    if MODEL_NAME == "graph_stid":
        model = ResidualGraphSTID(
            roads=N_ROADS,
            dynamic_dim=DYNAMIC_DIM,
            static_dim=STATIC_DIM,
            hidden_dim=CFG["HIDDEN_DIM"],
            road_embed_dim=CFG["ROAD_EMBED_DIM"],
            horizons=len(HORIZONS),
            dropout=CFG["DROPOUT"],
        )
    elif MODEL_NAME == "graph_wavenet":
        model = CompactGraphWaveNet(
            roads=N_ROADS,
            history=HIST_LEN,
            horizons=len(HORIZONS),
            dilations=(1, 2, 4, 8),
            residual_channels=32,
            skip_channels=64,
        )
    else:
        raise ValueError(f"Model tidak dikenal: {MODEL_NAME}")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {MODEL_NAME} | Parameter: {n_params:,}")

    ckpt_path = WORK_DIR / "outputs" / f"best_fold{fold_idx}_{MODEL_NAME}.pt"

    # ---- Pelatihan ----
    result = train_model(
        model=model,
        train_loader=loader_train,
        val_loader=loader_val,
        static_features=static_t,
        epochs=epochs,
        patience=CFG["PATIENCE"],
        amp=CFG["AMP"],
        learning_rate=CFG["LR"],
        weight_decay=CFG["WEIGHT_DECAY"],
        grad_clip=CFG["GRAD_CLIP"],
        checkpoint_path=ckpt_path,
        clip_range=tuple(CFG["PREDICTION_CLIP"]),
    )

    best_epochs_per_fold.append(result.best_epoch)

    # ---- Evaluasi per regime ----
    print(f"\nEvaluasi fold {fold_idx}:")
    res_m1 = evaluate_mse(
        model, speed_m1, va_m1, norm_m1, mean_m1, c_m1,
        CFG, "M1", always_zero_m1,
    )
    res_m2 = evaluate_mse(
        model, speed_m2, va_m2, norm_m2, mean_m2, c_m2,
        CFG, "M2", always_zero_m2,
    )

    # MSE gabungan dengan bobot regime
    combined_mse = (
        REGIME_WEIGHTS["m1"] * res_m1["overall"]
        + REGIME_WEIGHTS["m2"] * res_m2["overall"]
    )
    combined_baseline = (
        REGIME_WEIGHTS["m1"] * res_m1["baseline_overall"]
        + REGIME_WEIGHTS["m2"] * res_m2["baseline_overall"]
    )

    print(f"\n  Fold {fold_idx} GABUNGAN: MSE={combined_mse:.4f} | "
          f"Baseline={combined_baseline:.4f} | "
          f"Delta={combined_mse - combined_baseline:+.4f}")

    fold_info = {
        "fold": fold_idx,
        "model": MODEL_NAME,
        "best_epoch": result.best_epoch,
        "best_val_mse": result.best_val_mse,
        "combined_mse": combined_mse,
        "combined_baseline": combined_baseline,
        "beats_baseline": combined_mse < combined_baseline,
        "m1": res_m1,
        "m2": res_m2,
        "checkpoint": str(ckpt_path),
    }
    cv_results.append(fold_info)

# Simpan ringkasan CV
cv_output_path = WORK_DIR / "outputs" / "notebook_cv_results.json"
with open(cv_output_path, "w") as f:
    _json.dump({"cv_results": cv_results, "cfg": CFG}, f, indent=2)
print(f"\nHasil CV disimpan ke: {cv_output_path}")

Mode: smoke | Model: graph_stid | Folds: [2]

FOLD 2
Train windows: 512 | Val windows: 2372
Model: graph_stid | Parameter: 245,635
Epoch   0 | train_loss=42.1075 | val_mse=35.7012 * | lr=0.000505 | 80.2s
Epoch   1 | train_loss=41.3332 | val_mse=35.5630 * | lr=0.000010 | 79.0s
Model terbaik dimuat dari epoch 1, val_mse=35.5630

Evaluasi fold 2:
  [M1] Model MSE: 36.8249 | Baseline: 41.0860 | Delta: -4.2611
    h5: 32.1799 (baseline: 36.4286)
    h10: 37.4009 (baseline: 41.8522)
    h15: 40.8940 (baseline: 44.9773)
  [M2] Model MSE: 32.6936 | Baseline: 39.7498 | Delta: -7.0562
    h5: 28.0567 (baseline: 36.3198)
    h10: 32.8182 (baseline: 39.9823)
    h15: 37.2059 (baseline: 42.9473)

  Fold 2 GABUNGAN: MSE=35.5396 | Baseline=40.6703 | Delta=-5.1307

Hasil CV disimpan ke: /kaggle/working/outputs/notebook_cv_results.json


In [15]:
# =============================================================
# CELL 16 - RINGKASAN RANKING DAN ACCEPTANCE GATE
# =============================================================

print("=" * 60)
print("RINGKASAN CV DAN ACCEPTANCE GATE")
print("=" * 60)

if not cv_results:
    print("Tidak ada hasil CV. Jalankan Cell 15 terlebih dahulu.")
else:
    # Statistik fold
    all_mse = [r["combined_mse"] for r in cv_results]
    all_baseline = [r["combined_baseline"] for r in cv_results]
    n_beats_baseline = sum(1 for r in cv_results if r["beats_baseline"])
    mean_mse = float(np.mean(all_mse))
    std_mse = float(np.std(all_mse))

    print(f"\nFold MSE (gabungan M1+M2):")
    for r in cv_results:
        marker = "LULUS" if r["beats_baseline"] else "GAGAL"
        print(f"  Fold {r['fold']}: {r['combined_mse']:.4f} "
              f"(baseline: {r['combined_baseline']:.4f}) [{marker}]")

    print(f"\nRata-rata MSE: {mean_mse:.4f} +/- {std_mse:.4f}")
    print(f"Fold yang mengalahkan baseline: {n_beats_baseline}/{len(cv_results)}")

    # Per horizon
    if len(cv_results) > 0 and len(cv_results[0]["m1"]) > 0:
        print("\nMSE per horizon (rata-rata fold):")
        for h in HORIZONS:
            h_mse_m1 = np.mean([r["m1"].get(f"h{h}", 0) for r in cv_results])
            h_mse_m2 = np.mean([r["m2"].get(f"h{h}", 0) for r in cv_results])
            h_combined = REGIME_WEIGHTS["m1"] * h_mse_m1 + REGIME_WEIGHTS["m2"] * h_mse_m2
            h_bl_m1 = np.mean([r["m1"].get(f"h{h}_baseline", 0) for r in cv_results])
            h_bl_m2 = np.mean([r["m2"].get(f"h{h}_baseline", 0) for r in cv_results])
            h_bl_combined = REGIME_WEIGHTS["m1"] * h_bl_m1 + REGIME_WEIGHTS["m2"] * h_bl_m2
            print(f"  h{h}: {h_combined:.4f} (baseline: {h_bl_combined:.4f}) "
                  f"delta: {h_combined - h_bl_combined:+.4f}")

    # ---- Acceptance Gate (Section 8 dari plan) ----
    print("\n" + "-" * 40)
    print("ACCEPTANCE GATE")
    print("-" * 40)

    gate_results = {}

    # Syarat 1: Mean shadow MSE < 39.00
    gate_results["mse_below_39"] = mean_mse < 39.00
    print(f"[{'LULUS' if gate_results['mse_below_39'] else 'GAGAL'}] "
          f"1. MSE rata-rata ({mean_mse:.4f}) < 39.00")

    # Syarat 2: Mengalahkan baseline pada >= 2/3 fold
    gate_results["beats_baseline_2of3"] = n_beats_baseline >= 2
    print(f"[{'LULUS' if gate_results['beats_baseline_2of3'] else 'GAGAL'}] "
          f"2. Mengalahkan baseline pada {n_beats_baseline}/{len(cv_results)} fold (min 2)")

    # Syarat 3: MSE h15 tidak lebih buruk dari baseline
    if cv_results:
        h15_mse = np.mean([
            REGIME_WEIGHTS["m1"] * r["m1"].get("h15", float("inf"))
            + REGIME_WEIGHTS["m2"] * r["m2"].get("h15", float("inf"))
            for r in cv_results
        ])
        h15_bl = np.mean([
            REGIME_WEIGHTS["m1"] * r["m1"].get("h15_baseline", float("inf"))
            + REGIME_WEIGHTS["m2"] * r["m2"].get("h15_baseline", float("inf"))
            for r in cv_results
        ])
        gate_results["h15_not_worse"] = h15_mse <= h15_bl
        print(f"[{'LULUS' if gate_results['h15_not_worse'] else 'GAGAL'}] "
              f"3. MSE h15 ({h15_mse:.4f}) <= baseline h15 ({h15_bl:.4f})")
    else:
        gate_results["h15_not_worse"] = False

    # Syarat 4: MSE per regime tidak memburuk > 1%
    regime_ok = True
    if cv_results:
        for rk in ["m1", "m2"]:
            r_mse = np.mean([r[rk]["overall"] for r in cv_results])
            r_bl = np.mean([r[rk]["baseline_overall"] for r in cv_results])
            if r_mse > r_bl * 1.01:
                regime_ok = False
                print(f"  PERINGATAN: MSE {rk.upper()} ({r_mse:.4f}) memburuk > 1% dari baseline ({r_bl:.4f})")
    gate_results["regime_not_degraded"] = regime_ok
    print(f"[{'LULUS' if gate_results['regime_not_degraded'] else 'GAGAL'}] "
          f"4. MSE per regime tidak memburuk > 1%")

    # Syarat 5: Prediksi finite (diperiksa setelah inference)
    gate_results["predictions_finite"] = True  # akan diperbarui setelah cell 18
    print(f"[TERTUNDA] 5. Seluruh prediksi finite dan dalam rentang (diperiksa di Cell 18)")

    # Syarat 6: Variasi fold dilaporkan
    gate_results["fold_variance_reported"] = std_mse > 0
    print(f"[{'LULUS' if gate_results['fold_variance_reported'] else 'GAGAL'}] "
          f"6. Variasi fold dilaporkan: std={std_mse:.4f}")

    # Keputusan final
    all_clear = all(v for k, v in gate_results.items() if k != "predictions_finite")
    SUBMIT_CANDIDATE = all_clear

    print(f"\n{'=' * 40}")
    print(f"SUBMIT_CANDIDATE = {SUBMIT_CANDIDATE}")
    if not SUBMIT_CANDIDATE:
        failed = [k for k, v in gate_results.items() if not v and k != "predictions_finite"]
        print(f"Syarat yang belum terpenuhi: {failed}")
    print(f"{'=' * 40}")

RINGKASAN CV DAN ACCEPTANCE GATE

Fold MSE (gabungan M1+M2):
  Fold 2: 35.5396 (baseline: 40.6703) [LULUS]

Rata-rata MSE: 35.5396 +/- 0.0000
Fold yang mengalahkan baseline: 1/1

MSE per horizon (rata-rata fold):
  h5: 30.8971 (baseline: 36.3948) delta: -5.4976
  h10: 35.9752 (baseline: 41.2705) delta: -5.2953
  h15: 39.7466 (baseline: 44.3457) delta: -4.5992

----------------------------------------
ACCEPTANCE GATE
----------------------------------------
[LULUS] 1. MSE rata-rata (35.5396) < 39.00
[GAGAL] 2. Mengalahkan baseline pada 1/1 fold (min 2)
[LULUS] 3. MSE h15 (39.7466) <= baseline h15 (44.3457)
[LULUS] 4. MSE per regime tidak memburuk > 1%
[TERTUNDA] 5. Seluruh prediksi finite dan dalam rentang (diperiksa di Cell 18)
[GAGAL] 6. Variasi fold dilaporkan: std=0.0000

SUBMIT_CANDIDATE = False
Syarat yang belum terpenuhi: ['beats_baseline_2of3', 'fold_variance_reported']


In [16]:
# =============================================================
# CELL 17 - PELATIHAN FINAL
# =============================================================
#
# Hanya dijalankan jika RUN_MODE == "final"
# Melatih model pada seluruh data train menggunakan
# jumlah epoch median dari hasil CV.

FINAL_MODEL = None
FINAL_NORM_M1 = None
FINAL_NORM_M2 = None
FINAL_MEAN_M1 = None
FINAL_MEAN_M2 = None
FINAL_COEFFS_M1 = None
FINAL_COEFFS_M2 = None

if RUN_MODE == "final":
    print("=" * 60)
    print("PELATIHAN FINAL PADA SELURUH DATA TRAIN")
    print("=" * 60)

    # Tentukan jumlah epoch: median dari best_epoch per fold
    if best_epochs_per_fold:
        final_epochs = int(np.median(best_epochs_per_fold)) + 5  # sedikit lebih dari median
    else:
        final_epochs = CFG["EPOCHS"]
    print(f"Epoch final: {final_epochs} (dari median CV: {np.median(best_epochs_per_fold) if best_epochs_per_fold else 'N/A'})")

    # Fit normalizer dan koefisien dari SELURUH data train
    FINAL_NORM_M1 = SpeedNormalizer().fit(speed_m1)
    FINAL_NORM_M2 = SpeedNormalizer().fit(speed_m2)
    FINAL_MEAN_M1, FINAL_COEFFS_M1 = compute_mean_reversion_coeffs(
        speed_m1, np.arange(HIST_LEN - 1, len(speed_m1) - max(HORIZONS)), stride=30
    )
    FINAL_MEAN_M2, FINAL_COEFFS_M2 = compute_mean_reversion_coeffs(
        speed_m2, np.arange(HIST_LEN - 1, len(speed_m2) - max(HORIZONS)), stride=30
    )

    # Dataset final: seluruh data
    all_origins_m1 = np.arange(HIST_LEN - 1, len(speed_m1) - max(HORIZONS))
    all_origins_m2 = np.arange(HIST_LEN - 1, len(speed_m2) - max(HORIZONS))

    ds_final_m1 = LazyTrafficDataset(
        speed_m1, all_origins_m1, FINAL_NORM_M1, FINAL_MEAN_M1, FINAL_COEFFS_M1,
        A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN,
    )
    ds_final_m2 = LazyTrafficDataset(
        speed_m2, all_origins_m2, FINAL_NORM_M2, FINAL_MEAN_M2, FINAL_COEFFS_M2,
        A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN,
    )
    ds_final_all = ConcatDataset([ds_final_m1, ds_final_m2])

    loader_final = DataLoader(
        ds_final_all, batch_size=CFG["BATCH_SIZE"], shuffle=True,
        num_workers=CFG["NUM_WORKERS"], pin_memory=DEVICE.type == "cuda", drop_last=True,
    )

    # Bangun model final
    if MODEL_NAME == "graph_stid":
        FINAL_MODEL = ResidualGraphSTID(
            roads=N_ROADS,
            dynamic_dim=DYNAMIC_DIM,
            static_dim=STATIC_DIM,
            hidden_dim=CFG["HIDDEN_DIM"],
            road_embed_dim=CFG["ROAD_EMBED_DIM"],
            horizons=len(HORIZONS),
            dropout=CFG["DROPOUT"],
        )
    elif MODEL_NAME == "graph_wavenet":
        FINAL_MODEL = CompactGraphWaveNet(
            roads=N_ROADS, history=HIST_LEN, horizons=len(HORIZONS),
            dilations=(1, 2, 4, 8), residual_channels=32, skip_channels=64,
        )

    final_ckpt = WORK_DIR / "outputs" / f"final_{MODEL_NAME}.pt"
    static_t = torch.from_numpy(road_static).to(DEVICE)

    # Tidak ada validasi pada pelatihan final - gunakan dummy val loader
    # (DataLoader dari subset kecil validasi M2 terakhir sebagai monitor)
    dummy_val_origins = np.arange(
        len(speed_m2) - 100, len(speed_m2) - max(HORIZONS)
    )
    ds_dummy_val = LazyTrafficDataset(
        speed_m2, dummy_val_origins, FINAL_NORM_M2, FINAL_MEAN_M2, FINAL_COEFFS_M2,
        A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN,
    )
    loader_dummy_val = DataLoader(ds_dummy_val, batch_size=32, shuffle=False)

    final_result = train_model(
        model=FINAL_MODEL,
        train_loader=loader_final,
        val_loader=loader_dummy_val,
        static_features=static_t,
        epochs=final_epochs,
        patience=final_epochs,  # tidak ada early stopping pada final
        amp=CFG["AMP"],
        learning_rate=CFG["LR"],
        weight_decay=CFG["WEIGHT_DECAY"],
        grad_clip=CFG["GRAD_CLIP"],
        checkpoint_path=final_ckpt,
        clip_range=tuple(CFG["PREDICTION_CLIP"]),
    )

    print(f"\nPelatihan final selesai. Model disimpan ke: {final_ckpt}")
else:
    print(f"RUN_MODE='{RUN_MODE}', melewati pelatihan final.")
    print("Atur CFG['RUN_MODE'] = 'final' dan jalankan ulang untuk melatih pada seluruh data.")

RUN_MODE='smoke', melewati pelatihan final.
Atur CFG['RUN_MODE'] = 'final' dan jalankan ulang untuk melatih pada seluruh data.


In [17]:
# =============================================================
# CELL 18 - INFERENSI TEST DAN ROUTING REGIME
# =============================================================

def inference_test(
    model: nn.Module,
    test_hist_np: np.ndarray,
    test_regimes_np: np.ndarray,
    norm_m1: SpeedNormalizer,
    norm_m2: SpeedNormalizer,
    mean_m1: np.ndarray,
    mean_m2: np.ndarray,
    coeffs_m1: dict,
    coeffs_m2: dict,
    cfg: dict,
    always_zero_m1: np.ndarray,
    always_zero_m2: np.ndarray,
) -> np.ndarray:
    """Menjalankan inferensi test dengan routing per regime.

    Setiap sampel test diarahkan ke normalizer dan koefisien
    mean-reversion yang sesuai dengan regime-nya (M1 atau M2).

    Returns:
        pred_cube: (540, 3, 1260) float32, urutan horizon [h5, h10, h15]
    """
    N_test = len(test_hist_np)
    H = len(HORIZONS)
    N = N_ROADS
    pred_cube = np.zeros((N_test, H, N), dtype=np.float32)

    model.eval()
    static_t = torch.from_numpy(road_static).to(DEVICE)

    with torch.no_grad():
        for sample_idx in range(N_test):
            window = test_hist_np[sample_idx]  # (15, 1260)
            regime = test_regimes_np[sample_idx]

            # Pilih normalizer dan koefisien sesuai regime
            if regime == "m1":
                norm = norm_m1
                long_mean = mean_m1
                mr_coeffs = coeffs_m1
                az_mask = always_zero_m1
            else:
                norm = norm_m2
                long_mean = mean_m2
                mr_coeffs = coeffs_m2
                az_mask = always_zero_m2

            # Normalisasi window
            window_norm = norm.transform(window)  # (15, N)

            # Hitung fitur dinamis (sama dengan LazyTrafficDataset)
            last_val = window_norm[-1]
            mean3 = window_norm[-3:].mean(0)
            mean5 = window_norm[-5:].mean(0)
            mean15 = window_norm.mean(0)
            diff1 = window_norm[-1] - window_norm[-2]
            diff5 = window_norm[-1] - window_norm[-5]
            std5 = window_norm[-5:].std(0)
            std15 = window_norm.std(0)
            global_mean = mean15.mean() * np.ones(N, dtype=np.float32)

            scalar_feats = np.stack(
                [last_val, mean3, mean5, mean15, diff1, diff5, std5, std15, global_mean], axis=0
            )  # (9, N)

            hop_feats = []
            for A_f, A_b in zip(A_FWD_HOPS, A_BWD_HOPS):
                fwd = window_norm @ A_f.T  # (15, N)
                bwd = window_norm @ A_b.T
                hop_feats.extend([fwd[-1], bwd[-1]])  # ambil timestep terakhir

            hop_last = np.stack(hop_feats, axis=0)  # (n_hops*2, N)
            dynamic = np.concatenate([scalar_feats, hop_last], axis=0).T  # (N, D)

            # Baseline mean-reversion
            last_raw = window[-1]  # (N,)
            bl_preds = []
            for h in HORIZONS:
                c = mr_coeffs[h]
                p = last_raw + c * (long_mean - last_raw)
                bl_preds.append(p)
            baseline = np.stack(bl_preds, axis=1)  # (N, H)

            # Inferensi model
            dyn_t = torch.from_numpy(dynamic.astype(np.float32)).unsqueeze(0).to(DEVICE)  # (1, N, D)
            bl_t = torch.from_numpy(baseline.astype(np.float32)).unsqueeze(0).to(DEVICE)  # (1, N, H)

            pred = model(dyn_t, static_t, bl_t)  # (1, N, H)
            pred_np = pred.squeeze(0).cpu().numpy()  # (N, H)

            # Kliping dan mask nol
            pred_np = np.clip(pred_np, cfg["PREDICTION_CLIP"][0], cfg["PREDICTION_CLIP"][1])
            pred_np[az_mask] = 0.0

            # Format cube: horizon di dimensi 1
            pred_cube[sample_idx] = pred_np.T  # (H, N)

    return pred_cube


# Jalankan inferensi jika ada model
PRED_CUBE = None

if RUN_MODE == "final" and FINAL_MODEL is not None:
    print("Inferensi test menggunakan model FINAL...")
    norm_inf_m1 = FINAL_NORM_M1
    norm_inf_m2 = FINAL_NORM_M2
    mean_inf_m1 = FINAL_MEAN_M1
    mean_inf_m2 = FINAL_MEAN_M2
    coeffs_inf_m1 = FINAL_COEFFS_M1
    coeffs_inf_m2 = FINAL_COEFFS_M2
    infer_model = FINAL_MODEL
elif cv_results:
    print("Inferensi test menggunakan model dari fold terakhir CV...")
    # Muat model fold terakhir yang tersedia
    last_fold = FOLDS_TO_RUN[-1]
    ckpt_inf = WORK_DIR / "outputs" / f"best_fold{last_fold}_{MODEL_NAME}.pt"

    if MODEL_NAME == "graph_stid":
        infer_model = ResidualGraphSTID(
            roads=N_ROADS, dynamic_dim=DYNAMIC_DIM, static_dim=STATIC_DIM,
            hidden_dim=CFG["HIDDEN_DIM"], road_embed_dim=CFG["ROAD_EMBED_DIM"],
            horizons=len(HORIZONS), dropout=0.0,  # tidak ada dropout saat inferensi
        )
    else:
        infer_model = CompactGraphWaveNet(
            roads=N_ROADS, history=HIST_LEN, horizons=len(HORIZONS),
            dilations=(1, 2, 4, 8), residual_channels=32, skip_channels=64,
        )

    if ckpt_inf.exists():
        ckpt_data = torch.load(ckpt_inf, map_location=DEVICE)
        infer_model.load_state_dict(ckpt_data["model_state"])
        infer_model.to(DEVICE)
        print(f"Checkpoint dimuat dari: {ckpt_inf}")
    else:
        print(f"Checkpoint tidak ditemukan: {ckpt_inf}. Menggunakan model inisialisasi nol.")
        infer_model.to(DEVICE)

    # Gunakan normalizer dari fold terakhir
    tr_m1_inf, _ = FOLDS_M1[last_fold]
    tr_m2_inf, _ = FOLDS_M2[last_fold]
    norm_inf_m1 = SpeedNormalizer().fit(speed_m1[: tr_m1_inf[-1] + 1])
    norm_inf_m2 = SpeedNormalizer().fit(speed_m2[: tr_m2_inf[-1] + 1])
    mean_inf_m1, coeffs_inf_m1 = compute_mean_reversion_coeffs(speed_m1, tr_m1_inf, stride=30)
    mean_inf_m2, coeffs_inf_m2 = compute_mean_reversion_coeffs(speed_m2, tr_m2_inf, stride=30)
else:
    print("Tidak ada model tersedia. Menggunakan baseline mean-reversion sebagai fallback.")
    # Fallback: gunakan mean-reversion saja
    norm_inf_m1 = SpeedNormalizer().fit(speed_m1)
    norm_inf_m2 = SpeedNormalizer().fit(speed_m2)
    all_tr_m1 = np.arange(HIST_LEN - 1, len(speed_m1) - max(HORIZONS))
    all_tr_m2 = np.arange(HIST_LEN - 1, len(speed_m2) - max(HORIZONS))
    mean_inf_m1, coeffs_inf_m1 = compute_mean_reversion_coeffs(speed_m1, all_tr_m1, stride=30)
    mean_inf_m2, coeffs_inf_m2 = compute_mean_reversion_coeffs(speed_m2, all_tr_m2, stride=30)
    infer_model = None

# Jalankan inferensi
if infer_model is not None:
    t0 = time.time()
    PRED_CUBE = inference_test(
        model=infer_model,
        test_hist_np=test_hist,
        test_regimes_np=test_regimes,
        norm_m1=norm_inf_m1,
        norm_m2=norm_inf_m2,
        mean_m1=mean_inf_m1,
        mean_m2=mean_inf_m2,
        coeffs_m1=coeffs_inf_m1,
        coeffs_m2=coeffs_inf_m2,
        cfg=CFG,
        always_zero_m1=always_zero_m1,
        always_zero_m2=always_zero_m2,
    )
    print(f"Inferensi test selesai dalam {time.time() - t0:.1f} dtk")
    print(f"pred_cube shape: {PRED_CUBE.shape}")
    print(f"Prediksi finite: {np.isfinite(PRED_CUBE).all()}")
    print(f"Rentang prediksi: [{PRED_CUBE.min():.2f}, {PRED_CUBE.max():.2f}]")
else:
    # Fallback: mean-reversion saja
    print("Menghitung prediksi baseline mean-reversion...")
    PRED_CUBE = np.zeros((N_TEST, len(HORIZONS), N_ROADS), dtype=np.float32)
    for i in range(N_TEST):
        window = test_hist[i]   # (15, N)
        regime = test_regimes[i]
        long_mean = mean_inf_m1 if regime == "m1" else mean_inf_m2
        coeffs = coeffs_inf_m1 if regime == "m1" else coeffs_inf_m2
        last_raw = window[-1]
        for h_idx, h in enumerate(HORIZONS):
            c = coeffs[h]
            PRED_CUBE[i, h_idx, :] = last_raw + c * (long_mean - last_raw)
    PRED_CUBE = np.clip(PRED_CUBE, CFG["PREDICTION_CLIP"][0], CFG["PREDICTION_CLIP"][1])
    print(f"Baseline pred_cube shape: {PRED_CUBE.shape}")

# Update gate syarat 5
assert np.isfinite(PRED_CUBE).all(), "GAGAL: terdapat prediksi NaN atau Inf!"
assert (PRED_CUBE >= CFG["PREDICTION_CLIP"][0]).all() and (PRED_CUBE <= CFG["PREDICTION_CLIP"][1]).all(), \
    "GAGAL: prediksi di luar rentang yang diizinkan!"
print("Syarat 5 (prediksi finite dan dalam rentang): LULUS")

Inferensi test menggunakan model dari fold terakhir CV...
Checkpoint dimuat dari: /kaggle/working/outputs/best_fold2_graph_stid.pt
Inferensi test selesai dalam 3.8 dtk
pred_cube shape: (540, 3, 1260)
Prediksi finite: True
Rentang prediksi: [0.00, 104.16]
Syarat 5 (prediksi finite dan dalam rentang): LULUS


In [18]:
# =============================================================
# CELL 19 - ENSEMBLE OOF PER HORIZON
# =============================================================
#
# Mencari bobot ensemble optimal per horizon menggunakan OOF predictions.
# Hanya dijalankan jika mode CV dengan minimal 2 fold.

FINAL_PRED_CUBE = PRED_CUBE.copy()  # default: prediksi model tunggal

if len(cv_results) >= 2:
    print("=" * 60)
    print("ENSEMBLE OOF PER HORIZON")
    print("=" * 60)

    # Kandidat: [model, baseline]
    # Bobot nonnegatif yang berjumlah 1
    WEIGHT_GRID = np.arange(0.0, 1.01, 0.1)
    candidate_weights = [(w, 1.0 - w) for w in WEIGHT_GRID]

    print(f"Ruang pencarian: {len(candidate_weights)} kombinasi bobot (grid 0.1)")
    print("Memilih bobot ensemble menggunakan fold 0 dan 1 (jika tersedia)...")

    # Untuk setiap horizon, cari bobot yang meminimalkan MSE gabungan
    ensemble_weights = {}  # {h: (w_model, w_baseline)}

    for h_idx, h in enumerate(HORIZONS):
        best_mse = float("inf")
        best_w = (1.0, 0.0)  # default: pure model

        # Gunakan fold 0 dan 1 untuk seleksi bobot (fold 2 untuk verifikasi)
        selection_folds = [r for r in cv_results if r["fold"] in [0, 1]]

        if not selection_folds:
            selection_folds = cv_results  # fallback: gunakan semua fold

        for w_model, w_base in candidate_weights:
            mse_list = []
            for r in selection_folds:
                # MSE gabungan dengan bobot ini
                # Estimasi dari MSE per horizon yang sudah dihitung
                m1_model_mse = r["m1"].get(f"h{h}", float("inf"))
                m1_base_mse = r["m1"].get(f"h{h}_baseline", float("inf"))
                m2_model_mse = r["m2"].get(f"h{h}", float("inf"))
                m2_base_mse = r["m2"].get(f"h{h}_baseline", float("inf"))

                # Estimasi MSE ensemble (aproksimasi linier - tidak akurat untuk semua kombinasi)
                # Gunakan heuristic: ensemble MSE ~ w^2*model + (1-w)^2*baseline + cross
                # Untuk aproksimasi sederhana, gunakan nilai yang tersimpan
                est_m1 = w_model * m1_model_mse + w_base * m1_base_mse
                est_m2 = w_model * m2_model_mse + w_base * m2_base_mse
                est_combined = REGIME_WEIGHTS["m1"] * est_m1 + REGIME_WEIGHTS["m2"] * est_m2
                mse_list.append(est_combined)

            mean_est = float(np.mean(mse_list)) if mse_list else float("inf")
            if mean_est < best_mse:
                best_mse = mean_est
                best_w = (w_model, w_base)

        ensemble_weights[h] = best_w
        print(f"  h{h}: bobot model={best_w[0]:.1f}, bobot baseline={best_w[1]:.1f} "
              f"(est MSE={best_mse:.4f})")

    # Verifikasi pada fold 2 (jika tersedia)
    fold2_results = [r for r in cv_results if r["fold"] == 2]
    if fold2_results:
        r2 = fold2_results[0]
        print("\nVerifikasi ensemble pada fold 2:")
        for h_idx, h in enumerate(HORIZONS):
            w_m, w_b = ensemble_weights[h]
            m1_ens = w_m * r2["m1"].get(f"h{h}", 0) + w_b * r2["m1"].get(f"h{h}_baseline", 0)
            m2_ens = w_m * r2["m2"].get(f"h{h}", 0) + w_b * r2["m2"].get(f"h{h}_baseline", 0)
            ens_comb = REGIME_WEIGHTS["m1"] * m1_ens + REGIME_WEIGHTS["m2"] * m2_ens
            m1_pure = r2["m1"].get(f"h{h}", 0)
            m2_pure = r2["m2"].get(f"h{h}", 0)
            pure_comb = REGIME_WEIGHTS["m1"] * m1_pure + REGIME_WEIGHTS["m2"] * m2_pure
            diff = ens_comb - pure_comb
            print(f"  h{h}: model={pure_comb:.4f} -> ensemble={ens_comb:.4f} ({diff:+.4f})")

    # Terapkan ensemble ke prediksi test
    # Bangun prediksi baseline test untuk ensemble
    bl_test_cube = np.zeros((N_TEST, len(HORIZONS), N_ROADS), dtype=np.float32)
    for i in range(N_TEST):
        window = test_hist[i]
        regime = test_regimes[i]
        long_mean = mean_inf_m1 if regime == "m1" else mean_inf_m2
        coeffs = coeffs_inf_m1 if regime == "m1" else coeffs_inf_m2
        last_raw = window[-1]
        for h_idx, h in enumerate(HORIZONS):
            c = coeffs[h]
            bl_test_cube[i, h_idx, :] = last_raw + c * (long_mean - last_raw)

    # Ensemble per horizon
    ensemble_cube = np.zeros_like(PRED_CUBE)
    for h_idx, h in enumerate(HORIZONS):
        w_m, w_b = ensemble_weights[h]
        ensemble_cube[:, h_idx, :] = (
            w_m * PRED_CUBE[:, h_idx, :] + w_b * bl_test_cube[:, h_idx, :]
        )

    ensemble_cube = np.clip(ensemble_cube, CFG["PREDICTION_CLIP"][0], CFG["PREDICTION_CLIP"][1])
    FINAL_PRED_CUBE = ensemble_cube
    print(f"\nEnsemble diterapkan ke prediksi test.")
    print(f"Bobot ensemble: { {h: ensemble_weights[h] for h in HORIZONS} }")
else:
    print(f"Hanya {len(cv_results)} fold tersedia. Melewati ensemble (min 2 fold diperlukan).")
    print("Menggunakan prediksi model tunggal.")

print(f"\nFINAL_PRED_CUBE shape: {FINAL_PRED_CUBE.shape}")
assert np.isfinite(FINAL_PRED_CUBE).all(), "GAGAL: ensemble menghasilkan nilai NaN/Inf!"

Hanya 1 fold tersedia. Melewati ensemble (min 2 fold diperlukan).
Menggunakan prediksi model tunggal.

FINAL_PRED_CUBE shape: (540, 3, 1260)


In [19]:
# =============================================================
# CELL 20 - PEMBUATAN SUBMISSION DAN ASSERTIONS
# =============================================================

# Verifikasi urutan ID sample_submission
s_idx = np.repeat(np.arange(N_TEST), len(HORIZONS) * N_ROADS)
h_idx = np.tile(np.repeat([5, 10, 15], N_ROADS), N_TEST)
r_idx = np.tile(np.arange(N_ROADS), N_TEST * len(HORIZONS))
expected_ids = (
    "test_" + pd.Series(s_idx).astype(str).str.zfill(5)
    + "_h" + pd.Series(h_idx).astype(str)
    + "_r" + pd.Series(r_idx).astype(str)
)
assert (expected_ids.values == sample_sub["id"].values).all(), \
    "GAGAL: Urutan id submission tidak cocok dengan sample_submission.csv!"
print("Verifikasi urutan ID: LULUS")


def make_submission(
    pred_cube: np.ndarray,
    path: Path,
    clip_range: Tuple[float, float] = (0.0, 160.0),
) -> pd.DataFrame:
    """Membuat file submission.csv dari pred_cube.

    Args:
        pred_cube: (N_TEST, 3, N_ROADS) float32, urutan horizon [h5, h10, h15]
        path: lokasi file output
        clip_range: rentang kliping prediksi

    Returns:
        DataFrame submission
    """
    assert pred_cube.shape == (N_TEST, len(HORIZONS), N_ROADS), \
        f"Shape pred_cube salah: {pred_cube.shape}, diharapkan ({N_TEST}, {len(HORIZONS)}, {N_ROADS})"

    pred_clipped = np.clip(pred_cube, clip_range[0], clip_range[1])

    assert np.isfinite(pred_clipped).all(), "GAGAL: Terdapat prediksi NaN atau Inf!"
    assert (pred_clipped >= clip_range[0]).all() and (pred_clipped <= clip_range[1]).all(), \
        "GAGAL: Prediksi di luar rentang yang diizinkan!"

    sub = sample_sub.copy()
    sub["speed"] = pred_clipped.reshape(-1)

    # Verifikasi shape dan id final
    assert sub.shape == (N_TEST * len(HORIZONS) * N_ROADS, 2), \
        f"Shape submission salah: {sub.shape}"
    assert sub.columns.tolist() == ["id", "speed"], \
        f"Kolom submission salah: {sub.columns.tolist()}"
    assert (sub["id"].values == sample_sub["id"].values).all(), \
        "GAGAL: Urutan id tidak cocok!"
    assert np.isfinite(sub["speed"]).all(), "GAGAL: Terdapat speed NaN/Inf!"
    assert sub["speed"].between(clip_range[0], clip_range[1]).all(), \
        f"GAGAL: Speed di luar rentang [{clip_range[0]}, {clip_range[1]}]!"

    sub.to_csv(path, index=False)
    return sub


# Tentukan path submission
if CFG["MAKE_SUBMISSION"] or RUN_MODE in ["final", "cv"]:
    sub_path = WORK_DIR / "submission.csv"
else:
    sub_path = WORK_DIR / "submission_smoke.csv"

submission = make_submission(
    FINAL_PRED_CUBE,
    sub_path,
    clip_range=tuple(CFG["PREDICTION_CLIP"]),
)

print(f"\nSubmission disimpan ke: {sub_path}")
print(f"Jumlah baris: {len(submission):,}")
print(f"Rentang speed: [{submission['speed'].min():.2f}, {submission['speed'].max():.2f}]")
print(f"Distribusi speed:")
print(submission["speed"].describe().round(3).to_string())
print(f"\nContoh baris pertama dan terakhir:")
print(submission.head(3).to_string(index=False))
print("...")
print(submission.tail(3).to_string(index=False))

print("\nSeluruh assertions submission: LULUS")

Verifikasi urutan ID: LULUS

Submission disimpan ke: /kaggle/working/submission_smoke.csv
Jumlah baris: 2,041,200
Rentang speed: [0.00, 104.16]
Distribusi speed:
count    2041200.000
mean          53.175
std           19.453
min            0.000
25%           43.434
50%           54.584
75%           67.117
max          104.160

Contoh baris pertama dan terakhir:
              id     speed
test_00000_h5_r0 47.899418
test_00000_h5_r1 66.931015
test_00000_h5_r2 68.169861
...
                  id     speed
test_00539_h15_r1257 62.898510
test_00539_h15_r1258 68.259476
test_00539_h15_r1259  0.000000

Seluruh assertions submission: LULUS


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Ringkasan Hasil <a name="20"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Konfigurasi Run

| Parameter | Nilai |
|---|---|
| Model | _(diisi setelah run)_ |
| Mode | _(diisi setelah run)_ |
| Fold | _(diisi setelah run)_ |
| Epoch terbaik | _(diisi setelah run)_ |

## Hasil MSE Validasi

| | h5 | h10 | h15 | Rata-rata |
|---|---|---|---|---|
| Baseline mean-reversion | - | - | - | ~40-42 |
| Model | _(diisi setelah run)_ | _(diisi setelah run)_ | _(diisi setelah run)_ | _(diisi setelah run)_ |

## Status Acceptance Gate

| Syarat | Status |
|---|---|
| MSE < 39.00 | _(diisi setelah run)_ |
| Mengalahkan baseline >= 2/3 fold | _(diisi setelah run)_ |
| MSE h15 tidak lebih buruk dari baseline | _(diisi setelah run)_ |
| Regime tidak memburuk > 1% | _(diisi setelah run)_ |
| Prediksi finite dan dalam rentang | _(diisi setelah run)_ |
| Variasi fold dilaporkan | _(diisi setelah run)_ |

## Catatan

Ringkasan ini harus diperbarui dari keluaran aktual setelah notebook dieksekusi penuh.
Tidak ada hasil yang diklaim sebelum benar-benar dijalankan.

---

## Daftar Pustaka

Shi, H., Wang, P., Liu, H., & Wang, T. (2022). *Spatial-Temporal Identity: A Simple yet Effective Baseline for Multivariate Time Series Forecasting*. Proceedings of the 31st ACM International Conference on Information and Knowledge Management (CIKM 2022).

Wu, Z., Pan, S., Long, G., Jiang, J., & Zhang, C. (2019). *Graph WaveNet for Deep Spatial-Temporal Graph Modeling*. Proceedings of the 28th International Joint Conference on Artificial Intelligence (IJCAI 2019).

Harris, C. R., Millman, K. J., van der Walt, S. J., dkk. (2020). *Array programming with NumPy*. Nature, 585, 357-362.

Virtanen, P., Gommers, R., Oliphant, T. E., dkk. (2020). *SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python*. Nature Methods, 17, 261-272.

Paszke, A., Gross, S., Massa, F., dkk. (2019). *PyTorch: An Imperative Style, High-Performance Deep Learning Library*. Advances in Neural Information Processing Systems 32 (NeurIPS 2019).

---

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Terima Kasih! <a name=""></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

**Terima kasih kepada:**

Panitia **Datathon 2026** yang menyediakan *dataset* dan tantangan peramalan lalu lintas yang kaya sinyal.

Tim pengembang *open source*: NumPy, SciPy, PyTorch, dan pandas yang menjadi fondasi notebook ini.